<a href="https://colab.research.google.com/github/Aadarshmishra13/ai-driven-drug-discovery/blob/main/notebooks/01_EGFR_Target_and_ChEMBL_Data_Audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# EGFR Target Selection and ChEMBL Data Audit

**Project:** AI-Driven Drug Discovery

**Phase:** 1 — Target Selection and Data Audit

**Therapeutic Area:** Oncology

**Target:** Epidermal Growth Factor Receptor (EGFR)

**ChEMBL Target ID:** CHEMBL203

## Objective

To validate EGFR as a suitable therapeutic target for an ML-based drug discovery project and assess the availability, quality and distribution of experimentally measured compound bioactivity data from ChEMBL.

In [ ]:
!pip install -q pandas numpy requests matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

print("Environment setup successful!")

Environment setup successful!


In [ ]:
print("HTTP Status:", response.status_code)
print("Response URL:", response.url)
print("Content-Type:", response.headers.get("Content-Type"))

print("\nResponse JSON:")
print(response.json())

HTTP Status: 200
Response URL: https://www.ebi.ac.uk/chembl/api/data/target/CHEMBL203.json
Content-Type: application/json

Response JSON:
{'cross_references': [], 'organism': 'Homo sapiens', 'pref_name': 'Epidermal growth factor receptor', 'species_group_flag': False, 'target_chembl_id': 'CHEMBL203', 'target_components': [{'accession': 'P00533', 'component_description': 'Epidermal growth factor receptor', 'component_id': 147, 'component_type': 'PROTEIN', 'relationship': 'SINGLE PROTEIN', 'target_component_synonyms': [{'component_synonym': '2.7.10.1', 'syn_type': 'EC_NUMBER'}, {'component_synonym': 'EGFR', 'syn_type': 'GENE_SYMBOL'}, {'component_synonym': 'Epidermal growth factor receptor', 'syn_type': 'UNIPROT'}, {'component_synonym': 'ERBB', 'syn_type': 'GENE_SYMBOL_OTHER'}, {'component_synonym': 'ERBB1', 'syn_type': 'GENE_SYMBOL_OTHER'}, {'component_synonym': 'HER1', 'syn_type': 'GENE_SYMBOL_OTHER'}, {'component_synonym': 'Proto-oncogene c-ErbB-1', 'syn_type': 'UNIPROT'}, {'component

In [ ]:
BASE_URL = "https://www.ebi.ac.uk/chembl/api/data"
TARGET_ID = "CHEMBL203"

url = f"{BASE_URL}/target/{TARGET_ID}.json"

response = requests.get(url, timeout=30)
response.raise_for_status()

# ChEMBL returns target information directly
target_data = response.json()

print("EGFR TARGET INFORMATION")
print("=" * 50)

print("Target ID:", target_data.get("target_chembl_id"))
print("Target Name:", target_data.get("pref_name"))
print("Organism:", target_data.get("organism"))

print("\nTARGET COMPONENT DETAILS")
print("=" * 50)

for component in target_data.get("target_components", []):

    print("Component ID:", component.get("component_id"))
    print("Description:", component.get("component_description"))
    print("UniProt Accession:", component.get("accession"))
    print("Component Type:", component.get("component_type"))
    print("Relationship:", component.get("relationship"))

EGFR TARGET INFORMATION
Target ID: CHEMBL203
Target Name: Epidermal growth factor receptor
Organism: Homo sapiens

TARGET COMPONENT DETAILS
Component ID: 147
Description: Epidermal growth factor receptor
UniProt Accession: P00533
Component Type: PROTEIN
Relationship: SINGLE PROTEIN


In [ ]:

# Step 2: Retrieve EGFR IC50 activity data from ChEMBL

BASE_URL = "https://www.ebi.ac.uk/chembl/api/data"
TARGET_ID = "CHEMBL203"

activity_url = f"{BASE_URL}/activity.json"

params = {
    "target_chembl_id": TARGET_ID,
    "standard_type": "IC50",
    "standard_relation": "=",
    "standard_units": "nM",
    "confidence_score": 9,
    "limit": 1000
}

response = requests.get(
    activity_url,
    params=params,
    timeout=60
)

response.raise_for_status()

data = response.json()

activities = data.get("activities", [])

print("EGFR ACTIVITY DATA")
print("=" * 45)
print("HTTP Status:", response.status_code)
print("Records retrieved:", len(activities))
print("Total matching records:", data["page_meta"]["total_count"])


EGFR ACTIVITY DATA
HTTP Status: 200
Records retrieved: 1000
Total matching records: 19646


In [ ]:

from urllib.parse import urljoin
import time

def fetch_all_chembl_records(url, params):

    records = []

    session = requests.Session()

    # First API request
    response = session.get(
        url,
        params=params,
        timeout=60
    )
    response.raise_for_status()

    data = response.json()

    records.extend(data.get("activities", []))

    total_expected = data["page_meta"]["total_count"]

    next_page = data["page_meta"].get("next")

    # Retrieve remaining pages
    while next_page:

        next_url = urljoin(
            "https://www.ebi.ac.uk",
            next_page
        )

        time.sleep(0.2)

        response = session.get(
            next_url,
            timeout=60
        )
        response.raise_for_status()

        data = response.json()

        records.extend(data.get("activities", []))

        next_page = data["page_meta"].get("next")

    print("PAGINATION REPORT")
    print("=" * 45)
    print("Expected records:", total_expected)
    print("Downloaded records:", len(records))
    print("Records remaining:", total_expected - len(records))

    if len(records) != total_expected:
        print("WARNING: Download count does not match API total.")
    else:
        print("SUCCESS: All matching records retrieved.")

    return records


egfr_activities = fetch_all_chembl_records(
    activity_url,
    params
)


PAGINATION REPORT
Expected records: 19646
Downloaded records: 19646
Records remaining: 0
SUCCESS: All matching records retrieved.


In [ ]:
df = pd.DataFrame(egfr_activities)

print("DATASET OVERVIEW")
print("=" * 45)

print("Total activity records:", len(df))
print("Total columns:", len(df.columns))
print("Unique compounds:", df["molecule_chembl_id"].nunique())
print("Unique assays:", df["assay_chembl_id"].nunique())

print("\nDataset dimensions:", df.shape)

DATASET OVERVIEW
Total activity records: 19646
Total columns: 47
Unique compounds: 10853
Unique assays: 2044

Dataset dimensions: (19646, 47)


In [ ]:
important_columns = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "assay_type",
    "standard_type",
    "standard_value",
    "standard_units",
    "standard_relation",
    "pchembl_value",
    "confidence_score",
    "data_validity_comment",
    "potential_duplicate"
]

available_columns = [
    col for col in important_columns
    if col in df.columns
]

display(df[available_columns].head(10))

,molecule_chembl_id,assay_chembl_id,assay_type,standard_type,standard_value,standard_units,standard_relation,pchembl_value,data_validity_comment,potential_duplicate
0,CHEMBL68920,CHEMBL674637,B,IC50,41.0,nM,=,7.39,None,0
1,CHEMBL68920,CHEMBL621151,F,IC50,300.0,nM,=,6.52,None,0
2,CHEMBL68920,CHEMBL615325,F,IC50,7820.0,nM,=,5.11,None,0
3,CHEMBL69960,CHEMBL674637,B,IC50,170.0,nM,=,6.77,None,0
4,CHEMBL69960,CHEMBL621151,F,IC50,40.0,nM,=,7.40,None,0
5,CHEMBL69960,CHEMBL615325,F,IC50,440.0,nM,=,6.36,None,0
6,CHEMBL137635,CHEMBL677833,B,IC50,9300.0,nM,=,5.03,None,0
7,CHEMBL306988,CHEMBL674643,B,IC50,500000.0,nM,=,None,Outside typical range,0
8,CHEMBL66879,CHEMBL674643,B,IC50,3000000.0,nM,=,None,Outside typical range,0
9,CHEMBL77085,CHEMBL674643,B,IC50,96000.0,nM,=,4.02,None,0


In [ ]:
print("ASSAY TYPE DISTRIBUTION")
print("=" * 45)

print(df["assay_type"].value_counts(dropna=False))

print("\nDATA VALIDITY COMMENTS")
print("=" * 45)

print(df["data_validity_comment"].value_counts(dropna=False))

print("\nPOTENTIAL DUPLICATES")
print("=" * 45)

print(df["potential_duplicate"].value_counts(dropna=False))

print("\nMISSING VALUES IN IMPORTANT COLUMNS")
print("=" * 45)

audit_columns = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_value",
    "pchembl_value",
    "assay_type",
    "data_validity_comment"
]

display(df[audit_columns].isnull().sum().to_frame("Missing values"))

ASSAY TYPE DISTRIBUTION
assay_type
B    18381
F      745
A      498
T       22
Name: count, dtype: int64

DATA VALIDITY COMMENTS
data_validity_comment
None                             19381
Outside typical range              206
Potential transcription error       59
Name: count, dtype: int64

POTENTIAL DUPLICATES
potential_duplicate
0    17463
1     2183
Name: count, dtype: int64

MISSING VALUES IN IMPORTANT COLUMNS


,Missing values
molecule_chembl_id,0
assay_chembl_id,0
standard_value,3
pchembl_value,268
assay_type,0
data_validity_comment,19381


In [ ]:
# Convert activity values to numeric
df["standard_value"] = pd.to_numeric(
    df["standard_value"],
    errors="coerce"
)

df["pchembl_value"] = pd.to_numeric(
    df["pchembl_value"],
    errors="coerce"
)

# Calculate pIC50 from IC50 values in nM
df["pIC50"] = np.where(
    df["standard_value"] > 0,
    9 - np.log10(df["standard_value"]),
    np.nan
)

print("pIC50 CALCULATION REPORT")
print("=" * 45)

print("Total records:", len(df))
print("Valid pIC50 values:", df["pIC50"].notna().sum())
print("Missing pIC50 values:", df["pIC50"].isna().sum())

print("\nPIC50 DESCRIPTIVE STATISTICS")
print("=" * 45)

display(df["pIC50"].describe())

print("\nCOMPARISON WITH ORIGINAL pChEMBL VALUES")
print("=" * 45)

display(
    df[["standard_value", "pchembl_value", "pIC50"]].head(15)
)

pIC50 CALCULATION REPORT
Total records: 19646
Valid pIC50 values: 19643
Missing pIC50 values: 3

PIC50 DESCRIPTIVE STATISTICS


,pIC50
count,19643.000000
mean,7.058429
std,1.382552
min,1.080922
25%,6.112034
50%,7.154902
75%,8.091515
max,17.299989



COMPARISON WITH ORIGINAL pChEMBL VALUES


,standard_value,pchembl_value,pIC50
0,41.0,7.39,7.387216
1,300.0,6.52,6.522879
2,7820.0,5.11,5.106793
3,170.0,6.77,6.769551
4,40.0,7.40,7.397940
5,440.0,6.36,6.356547
6,9300.0,5.03,5.031517
7,500000.0,NaN,3.301030
8,3000000.0,NaN,2.522879
9,96000.0,4.02,4.017729


In [ ]:
# Highest pIC50 values
print("TOP 15 HIGHEST pIC50 RECORDS")
print("=" * 70)

display(
    df.sort_values("pIC50", ascending=False)[
        [
            "molecule_chembl_id",
            "assay_chembl_id",
            "assay_type",
            "standard_value",
            "standard_units",
            "pchembl_value",
            "pIC50",
            "data_validity_comment",
            "potential_duplicate"
        ]
    ].head(15)
)


# Lowest pIC50 values
print("\nTOP 15 LOWEST pIC50 RECORDS")
print("=" * 70)

display(
    df.sort_values("pIC50", ascending=True)[
        [
            "molecule_chembl_id",
            "assay_chembl_id",
            "assay_type",
            "standard_value",
            "standard_units",
            "pchembl_value",
            "pIC50",
            "data_validity_comment",
            "potential_duplicate"
        ]
    ].head(15)
)

TOP 15 HIGHEST pIC50 RECORDS


,molecule_chembl_id,assay_chembl_id,assay_type,standard_value,standard_units,pchembl_value,pIC50,data_validity_comment,potential_duplicate
11416,CHEMBL5073622,CHEMBL5031734,B,5.012000e-09,nM,NaN,17.299989,Outside typical range,0
10876,CHEMBL3353410,CHEMBL4721173,B,2.000000e-03,nM,NaN,11.698970,Potential transcription error,1
1596,CHEMBL63786,CHEMBL683040,B,3.000000e-03,nM,NaN,11.522879,Outside typical range,0
1085,CHEMBL53711,CHEMBL674648,B,6.000000e-03,nM,NaN,11.221849,Outside typical range,1
1527,CHEMBL35820,CHEMBL680017,B,6.000000e-03,nM,NaN,11.221849,Outside typical range,0
1711,CHEMBL53711,CHEMBL680021,B,6.000000e-03,nM,NaN,11.221849,Outside typical range,0
224,CHEMBL53711,CHEMBL680030,F,6.000000e-03,nM,NaN,11.221849,Outside typical range,1
2992,CHEMBL35820,CHEMBL939337,B,6.000000e-03,nM,NaN,11.221849,Outside typical range,1
331,CHEMBL35820,CHEMBL677389,B,6.000000e-03,nM,NaN,11.221849,Outside typical range,1
2939,CHEMBL35820,CHEMBL952131,B,6.026000e-03,nM,NaN,11.219971,Outside typical range,1



TOP 15 LOWEST pIC50 RECORDS


,molecule_chembl_id,assay_chembl_id,assay_type,standard_value,standard_units,pchembl_value,pIC50,data_validity_comment,potential_duplicate
13383,CHEMBL607707,CHEMBL5386735,B,83000000.00,nM,NaN,1.080922,Potential transcription error,1
11097,CHEMBL4798627,CHEMBL4732637,B,55000000.00,nM,NaN,1.259637,Outside typical range,0
3495,CHEMBL45068,CHEMBL970971,B,25000000.00,nM,NaN,1.602060,Outside typical range,0
3494,CHEMBL490510,CHEMBL970971,B,19000000.00,nM,NaN,1.721246,Outside typical range,0
5112,CHEMBL113996,CHEMBL3056386,B,13567505.30,nM,NaN,1.867500,Outside typical range,0
6202,CHEMBL3353400,CHEMBL3389022,B,11000000.00,nM,NaN,1.958607,Outside typical range,0
5108,CHEMBL114073,CHEMBL3056386,B,10590098.51,nM,NaN,1.975100,Outside typical range,0
5119,CHEMBL116012,CHEMBL3056386,B,10320482.46,nM,NaN,1.986300,Outside typical range,0
6205,CHEMBL3353401,CHEMBL3389022,B,10300000.00,nM,NaN,1.987163,Outside typical range,0
6187,CHEMBL3353395,CHEMBL3389022,B,10000000.00,nM,NaN,2.000000,Outside typical range,0


In [ ]:
print("RECORDS WITH DATA VALIDITY WARNINGS")
print("=" * 60)

display(
    df[df["data_validity_comment"].notna()][
        [
            "molecule_chembl_id",
            "assay_chembl_id",
            "standard_value",
            "pchembl_value",
            "pIC50",
            "data_validity_comment"
        ]
    ].head(20)
)

RECORDS WITH DATA VALIDITY WARNINGS


,molecule_chembl_id,assay_chembl_id,standard_value,pchembl_value,pIC50,data_validity_comment
7,CHEMBL306988,CHEMBL674643,500000.0,NaN,3.301030,Outside typical range
8,CHEMBL66879,CHEMBL674643,3000000.0,NaN,2.522879,Outside typical range
11,CHEMBL76979,CHEMBL674643,264000.0,NaN,3.578396,Outside typical range
14,CHEMBL1009,CHEMBL674643,900000.0,NaN,3.045757,Outside typical range
18,CHEMBL77825,CHEMBL674643,600000.0,NaN,3.221849,Outside typical range
19,CHEMBL421877,CHEMBL674643,850000.0,NaN,3.070581,Outside typical range
22,CHEMBL309866,CHEMBL674643,1300000.0,NaN,2.886057,Outside typical range
23,CHEMBL77381,CHEMBL674643,1360000.0,NaN,2.866461,Outside typical range
24,CHEMBL76602,CHEMBL674643,165000.0,NaN,3.782516,Outside typical range
25,CHEMBL77737,CHEMBL674643,250000.0,NaN,3.602060,Outside typical range


In [ ]:
# Inspect all available fields
print("AVAILABLE DATASET COLUMNS")
print("=" * 60)

for col in df.columns:
    print(col)

AVAILABLE DATASET COLUMNS
action_type
activity_comment
activity_id
activity_properties
assay_chembl_id
assay_description
assay_type
assay_variant_accession
assay_variant_mutation
bao_endpoint
bao_format
bao_label
canonical_smiles
data_validity_comment
data_validity_description
document_chembl_id
document_journal
document_year
ligand_efficiency
modality
molecule_chembl_id
molecule_pref_name
parent_molecule_chembl_id
pchembl_value
potential_duplicate
qudt_units
record_id
relation
src_id
standard_flag
standard_relation
standard_text_value
standard_type
standard_units
standard_upper_value
standard_value
target_chembl_id
target_organism
target_pref_name
target_tax_id
text_value
toid
type
units
uo_units
upper_value
value
pIC50


In [ ]:
# Inspect metadata for representative assays
metadata_columns = [
    "assay_chembl_id",
    "assay_type",
    "assay_description",
    "assay_organism",
    "assay_tissue",
    "assay_cell_type",
    "assay_variant_accession",
    "assay_variant_mutation",
    "target_organism",
    "target_pref_name",
    "bao_label",
    "document_chembl_id"
]

available_metadata = [
    col for col in metadata_columns
    if col in df.columns
]

assay_metadata = (
    df[available_metadata]
    .drop_duplicates(subset=["assay_chembl_id"])
)

print("UNIQUE ASSAY METADATA")
print("=" * 60)
print("Unique assays:", len(assay_metadata))

display(assay_metadata.head(20))

UNIQUE ASSAY METADATA
Unique assays: 2044


,assay_chembl_id,assay_type,assay_description,assay_variant_accession,assay_variant_mutation,target_organism,target_pref_name,bao_label,document_chembl_id
0,CHEMBL674637,B,Inhibitory activity towards tyrosine phosphorylation for the epidermal growth factor-receptor ki...,None,None,Homo sapiens,Epidermal growth factor receptor,single protein format,CHEMBL1134862
1,CHEMBL621151,F,Inhibition of autophosphorylation of human epidermal growth factor-receptor (EGF-R) expressed in...,None,None,Homo sapiens,Epidermal growth factor receptor,cell-based format,CHEMBL1134862
2,CHEMBL615325,F,Inhibition of ligand-induced proliferation in human EGF-R Kinase expressing 3T3 cells,None,None,Homo sapiens,Epidermal growth factor receptor,cell-based format,CHEMBL1134862
6,CHEMBL677833,B,In vitro inhibition of Epidermal growth factor receptor tyrosine kinase isolated from A431 (huma...,None,None,Homo sapiens,Epidermal growth factor receptor,single protein format,CHEMBL1145114
7,CHEMBL674643,B,Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor recepto...,None,None,Homo sapiens,Epidermal growth factor receptor,single protein format,CHEMBL1124610
89,CHEMBL679674,B,Inhibition of Epidermal growth factor receptor,None,None,Homo sapiens,Epidermal growth factor receptor,single protein format,CHEMBL1132739
127,CHEMBL679949,B,Evaluated for inhibitory activity towards tyrosine kinase Epidermal growth factor receptor,None,None,Homo sapiens,Epidermal growth factor receptor,single protein format,CHEMBL1133768
128,CHEMBL674698,B,Inhibition of Epidermal growth factor receptor,None,None,Homo sapiens,Epidermal growth factor receptor,single protein format,CHEMBL1134917
131,CHEMBL673925,B,Inhibition of phosphorylation of a polyglutamic acid/tyrosine random copolymer by EGFR enzyme (f...,None,None,Homo sapiens,Epidermal growth factor receptor,single protein format,CHEMBL1134843
132,CHEMBL677216,B,Inhibition of autophosphorylation of EGFR in A431 cells,None,None,Homo sapiens,Epidermal growth factor receptor,cell-based format,CHEMBL1134843


In [ ]:
extreme_record = df.loc[
    df["molecule_chembl_id"] == "CHEMBL5073622"
]

display(extreme_record.T)

,11416
action_type,"{'action_type': 'INHIBITOR', 'description': 'Negatively effects (inhibits) the normal functionin..."
activity_comment,None
activity_id,24348978
activity_properties,[]
assay_chembl_id,CHEMBL5031734
assay_description,Inhibition of human EGFR
assay_type,B
assay_variant_accession,None
assay_variant_mutation,None
bao_endpoint,BAO_0000190


In [ ]:
# Inspect the extreme record at full numerical precision

extreme = df.loc[
    df["molecule_chembl_id"] == "CHEMBL5073622"
].iloc[0]

print("EXTREME ACTIVITY RECORD")
print("=" * 55)

print("Compound:", extreme["molecule_chembl_id"])
print("Compound name:", extreme["molecule_pref_name"])
print("Assay:", extreme["assay_chembl_id"])
print("Standard value:", repr(extreme["standard_value"]))
print("Standard value (precision):", format(extreme["standard_value"], ".15g"))
print("Calculated pIC50:", extreme["pIC50"])
print("Original pChEMBL:", extreme["pchembl_value"])
print("Validity comment:", extreme["data_validity_comment"])

print("\nRecalculated pIC50:")

if extreme["standard_value"] > 0:
    print(9 - np.log10(extreme["standard_value"]))
else:
    print("Cannot calculate: value is zero or negative.")

EXTREME ACTIVITY RECORD
Compound: CHEMBL5073622
Compound name: TAK-020
Assay: CHEMBL5031734
Standard value: np.float64(5.012e-09)
Standard value (precision): 5.012e-09
Calculated pIC50: 17.299988937677888
Original pChEMBL: nan
Validity comment: Outside typical range

Recalculated pIC50:
17.299988937677888


In [ ]:
print("ASSAY TYPE DISTRIBUTION")
display(
    assay_metadata["assay_type"]
    .value_counts(dropna=False)
    .to_frame("Unique assays")
)

print("\nBIOLOGICAL ASSAY FORMAT DISTRIBUTION")
display(
    assay_metadata["bao_label"]
    .value_counts(dropna=False)
    .to_frame("Unique assays")
)

print("\nASSAY TYPE × BIOLOGICAL FORMAT")
display(
    pd.crosstab(
        assay_metadata["assay_type"],
        assay_metadata["bao_label"],
        margins=True
    )
)

ASSAY TYPE DISTRIBUTION


,Unique assays
assay_type,
B,1910
F,82
A,50
T,2



BIOLOGICAL ASSAY FORMAT DISTRIBUTION


,Unique assays
bao_label,
single protein format,1215
cell-based format,557
assay format,207
protein format,55
cell membrane format,8
cell-free format,1
organism-based format,1



ASSAY TYPE × BIOLOGICAL FORMAT


bao_label,assay format,cell membrane format,cell-based format,cell-free format,organism-based format,protein format,single protein format,All
assay_type,,,,,,,,
A,17,0,12,0,0,0,21,50
B,168,8,483,1,1,55,1194,1910
F,21,0,61,0,0,0,0,82
T,1,0,1,0,0,0,0,2
All,207,8,557,1,1,55,1215,2044


In [ ]:
mutation_keywords = (
    r"mutant|mutation|L858R|T790M|C797S|exon 19|deletion"
)

mutation_assays = assay_metadata[
    assay_metadata["assay_description"]
    .str.contains(mutation_keywords, case=False, na=False)
]

print("ASSAYS WITH POTENTIAL MUTATION REFERENCES")
print("Count:", len(mutation_assays))

display(
    mutation_assays[
        [
            "assay_chembl_id",
            "assay_type",
            "assay_description",
            "assay_variant_mutation"
        ]
    ].head(30)
)

ASSAYS WITH POTENTIAL MUTATION REFERENCES
Count: 649


,assay_chembl_id,assay_type,assay_description,assay_variant_mutation
2990,CHEMBL939338,B,Inhibition of EGFR L858R mutant,L858R
2991,CHEMBL939339,B,Inhibition of EGFR L861Q mutant,L861Q
3211,CHEMBL964130,B,Inhibition of EGFR L858R mutant by time resolved fluorescence assay,L858R
3542,CHEMBL1044057,B,Inhibition of Her1 T790M mutant by fluorescence polarization assay,T790M
3792,CHEMBL1108837,B,Inhibition of EGFR L858R mutant by time resolved fluorescence assay,L858R
3822,CHEMBL1105218,B,Inhibition of EGFR Leu858Arg and Thr790Met mutant by HTRF assay,"L858R,T790M"
3831,CHEMBL1105217,B,Inhibition of EGFR Leu858Arg mutant by HTRF assay,L858R
3961,CHEMBL1176123,B,Inhibition of EGFR L858R/T790M double mutant,"L858R,T790M"
4149,CHEMBL1244583,B,Inhibition of EGFR T790M mutant,T790M
4191,CHEMBL1273339,B,Inhibition of EGFR L858R/T790M double mutant,"L858R,T790M"


In [ ]:
# Create a working copy for assay-level classification
audit_df = df.copy()

# Identify assays with explicit mutation annotations
mutation_pattern = (
    r"mutant|mutation|L858R|T790M|C797S|L861Q|"
    r"exon 19|deletion|L834R"
)

audit_df["mutation_flag"] = (
    audit_df["assay_variant_mutation"].notna()
    | audit_df["assay_description"].str.contains(
        mutation_pattern,
        case=False,
        na=False
    )
)

print("ACTIVITY RECORD DISTRIBUTION")
print("=" * 55)

print("Total records:", len(audit_df))
print("Records with mutation references:",
      audit_df["mutation_flag"].sum())
print("Records without explicit mutation references:",
      (~audit_df["mutation_flag"]).sum())

print("\nRECORDS BY ASSAY TYPE")
display(
    audit_df.groupby("assay_type")
    .size()
    .to_frame("Activity records")
)

print("\nRECORDS BY BIOLOGICAL FORMAT")
display(
    audit_df.groupby("bao_label")
    .size()
    .sort_values(ascending=False)
    .to_frame("Activity records")
)

print("\nMUTATION STATUS × ASSAY TYPE")
display(
    pd.crosstab(
        audit_df["mutation_flag"],
        audit_df["assay_type"],
        margins=True
    )
)

ACTIVITY RECORD DISTRIBUTION
Total records: 19646
Records with mutation references: 6368
Records without explicit mutation references: 13278

RECORDS BY ASSAY TYPE


,Activity records
assay_type,
A,498
B,18381
F,745
T,22



RECORDS BY BIOLOGICAL FORMAT


,Activity records
bao_label,
single protein format,11746
cell-based format,4914
assay format,2420
protein format,495
cell membrane format,40
cell-free format,27
organism-based format,4



MUTATION STATUS × ASSAY TYPE


assay_type,A,B,F,T,All
mutation_flag,,,,,
False,498,12013,745,22,13278
True,0,6368,0,0,6368
All,498,18381,745,22,19646


In [ ]:
print("MUTATION ANNOTATION DISTRIBUTION")
print("=" * 55)

display(
    audit_df.loc[
        audit_df["mutation_flag"],
        "assay_variant_mutation"
    ]
    .fillna("Description-only annotation")
    .value_counts()
    .head(20)
)

MUTATION ANNOTATION DISTRIBUTION


,count
assay_variant_mutation,
Description-only annotation,2102
"L858R,T790M",1640
UNDEFINED MUTATION,823
L858R,734
"L858R,T790M,C797S",673
T790M,246
L861Q,45
"C797S,L858R",42
"T790M,L858M",30


In [ ]:
candidate_records = audit_df[
    (audit_df["assay_type"] == "B")
    & (audit_df["bao_label"] == "single protein format")
    & (~audit_df["mutation_flag"])
]

print("INITIAL CANDIDATE RECORDS")
print("=" * 55)

print("Candidate activity records:", len(candidate_records))
print(
    "Unique compounds:",
    candidate_records["molecule_chembl_id"].nunique()
)
print(
    "Unique assays:",
    candidate_records["assay_chembl_id"].nunique()
)

display(
    candidate_records[
        [
            "molecule_chembl_id",
            "assay_chembl_id",
            "assay_description",
            "standard_value",
            "pIC50",
            "data_validity_comment",
            "potential_duplicate"
        ]
    ].head(10)
)

INITIAL CANDIDATE RECORDS
Candidate activity records: 8082
Unique compounds: 6217
Unique assays: 853


,molecule_chembl_id,assay_chembl_id,assay_description,standard_value,pIC50,data_validity_comment,potential_duplicate
0,CHEMBL68920,CHEMBL674637,Inhibitory activity towards tyrosine phosphorylation for the epidermal growth factor-receptor ki...,41.0,7.387216,None,0
3,CHEMBL69960,CHEMBL674637,Inhibitory activity towards tyrosine phosphorylation for the epidermal growth factor-receptor ki...,170.0,6.769551,None,0
6,CHEMBL137635,CHEMBL677833,In vitro inhibition of Epidermal growth factor receptor tyrosine kinase isolated from A431 (huma...,9300.0,5.031517,None,0
7,CHEMBL306988,CHEMBL674643,Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor recepto...,500000.0,3.301030,Outside typical range,0
8,CHEMBL66879,CHEMBL674643,Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor recepto...,3000000.0,2.522879,Outside typical range,0
9,CHEMBL77085,CHEMBL674643,Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor recepto...,96000.0,4.017729,None,0
10,CHEMBL443268,CHEMBL674637,Inhibitory activity towards tyrosine phosphorylation for the epidermal growth factor-receptor ki...,5310.0,5.274905,None,0
11,CHEMBL76979,CHEMBL674643,Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor recepto...,264000.0,3.578396,Outside typical range,0
12,CHEMBL76589,CHEMBL674643,Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor recepto...,125.0,6.903090,None,0
13,CHEMBL76904,CHEMBL674643,Inhibitory concentration of EGF dependent autophosphorylation of epidermal growth factor recepto...,35000.0,4.455932,None,0


In [ ]:
# Create a separate candidate dataset
candidate_df = candidate_records.copy()

# Apply quality checks independently
quality_checks = {
    "Initial candidate records": pd.Series(
        True, index=candidate_df.index
    ),
    "No validity warning": (
        candidate_df["data_validity_comment"].isna()
    ),
    "No potential duplicate flag": (
        candidate_df["potential_duplicate"] == 0
    ),
    "Positive IC50 value": (
        candidate_df["standard_value"] > 0
    ),
    "Available canonical SMILES": (
        candidate_df["canonical_smiles"].notna()
    ),
    "Available pIC50": (
        candidate_df["pIC50"].notna()
    )
}

print("DATA QUALITY AUDIT")
print("=" * 60)

for criterion, mask in quality_checks.items():
    print(f"{criterion}: {mask.sum():,}")

DATA QUALITY AUDIT
Initial candidate records: 8,082
No validity warning: 7,949
No potential duplicate flag: 7,118
Positive IC50 value: 8,081
Available canonical SMILES: 8,073
Available pIC50: 8,081


In [ ]:
clean_mask = (
    candidate_df["data_validity_comment"].isna()
    & (candidate_df["potential_duplicate"] == 0)
    & (candidate_df["standard_value"] > 0)
    & candidate_df["canonical_smiles"].notna()
    & candidate_df["pIC50"].notna()
)

clean_candidates = candidate_df.loc[clean_mask].copy()

print("INITIAL CLEAN DATASET")
print("=" * 55)

print("Activity records:", len(clean_candidates))
print(
    "Unique compounds:",
    clean_candidates["molecule_chembl_id"].nunique()
)
print(
    "Unique assays:",
    clean_candidates["assay_chembl_id"].nunique()
)

print("\npIC50 statistics:")
display(clean_candidates["pIC50"].describe().to_frame())

INITIAL CLEAN DATASET
Activity records: 7000
Unique compounds: 5938
Unique assays: 752

pIC50 statistics:


,pIC50
count,7000.000000
mean,7.118543
std,1.357325
min,4.000000
25%,6.111579
50%,7.200659
75%,8.200659
max,11.000000


In [ ]:
print("DUPLICATE FLAG AUDIT")
print("=" * 55)

display(
    candidate_df.groupby("potential_duplicate")
    .agg(
        activity_records=("activity_id", "count"),
        unique_compounds=("molecule_chembl_id", "nunique"),
        unique_assays=("assay_chembl_id", "nunique")
    )
)

DUPLICATE FLAG AUDIT


,activity_records,unique_compounds,unique_assays
potential_duplicate,,,
0,7118,6022,759
1,964,627,324


In [ ]:
# Isolate records flagged as potential duplicates
duplicate_df = candidate_df[
    candidate_df["potential_duplicate"] == 1
].copy()

# Examine repeated compound-assay combinations
duplicate_groups = (
    duplicate_df
    .groupby(["molecule_chembl_id", "assay_chembl_id"])
    .agg(
        record_count=("activity_id", "count"),
        unique_ic50_values=("standard_value", "nunique"),
        unique_documents=("document_chembl_id", "nunique"),
        unique_relations=("standard_relation", "nunique")
    )
    .reset_index()
)

print("DUPLICATE RECORD INVESTIGATION")
print("=" * 60)

print("Flagged activity records:", len(duplicate_df))
print("Unique compound-assay combinations:", len(duplicate_groups))

print(
    "Repeated compound-assay combinations:",
    (duplicate_groups["record_count"] > 1).sum()
)

print(
    "Combinations with multiple IC50 values:",
    (duplicate_groups["unique_ic50_values"] > 1).sum()
)

print(
    "Combinations reported in multiple documents:",
    (duplicate_groups["unique_documents"] > 1).sum()
)

print("\nEXAMPLE FLAGGED GROUPS")
display(
    duplicate_groups
    .sort_values("record_count", ascending=False)
    .head(15)
)

DUPLICATE RECORD INVESTIGATION
Flagged activity records: 964
Unique compound-assay combinations: 956
Repeated compound-assay combinations: 7
Combinations with multiple IC50 values: 2
Combinations reported in multiple documents: 0

EXAMPLE FLAGGED GROUPS


,molecule_chembl_id,assay_chembl_id,record_count,unique_ic50_values,unique_documents,unique_relations
674,CHEMBL5744203,CHEMBL5735011,3,1,1,1
30,CHEMBL1173655,CHEMBL5230889,2,2,1,1
118,CHEMBL1873475,CHEMBL3888240,2,1,1,1
761,CHEMBL5916409,CHEMBL5735011,2,1,1,1
783,CHEMBL5952540,CHEMBL5735011,2,1,1,1
338,CHEMBL3647967,CHEMBL3888240,2,1,1,1
678,CHEMBL5752119,CHEMBL5735965,2,2,1,1
638,CHEMBL554,CHEMBL5150423,1,1,1,1
637,CHEMBL554,CHEMBL5138507,1,1,1,1
636,CHEMBL554,CHEMBL4729717,1,1,1,1


In [ ]:
# Identify compound-assay combinations with repeated records
repeated_pairs = duplicate_groups[
    duplicate_groups["record_count"] > 1
][["molecule_chembl_id", "assay_chembl_id"]]

# Retrieve the original activity records
repeated_records = duplicate_df.merge(
    repeated_pairs,
    on=["molecule_chembl_id", "assay_chembl_id"],
    how="inner"
)

columns_to_inspect = [
    "activity_id",
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_value",
    "standard_units",
    "standard_relation",
    "pIC50",
    "document_chembl_id",
    "assay_description",
    "data_validity_comment"
]

display(
    repeated_records[columns_to_inspect]
    .sort_values(
        ["molecule_chembl_id", "assay_chembl_id"]
    )
    .reset_index(drop=True)
)

,activity_id,molecule_chembl_id,assay_chembl_id,standard_value,standard_units,standard_relation,pIC50,document_chembl_id,assay_description,data_validity_comment
0,24993700,CHEMBL1173655,CHEMBL5230889,4.0,nM,=,8.397940,CHEMBL5230100,Inhibition of EGFR (unknown origin) using FAM-labeled peptide as substrate incubated for 10 mins...,None
1,24993702,CHEMBL1173655,CHEMBL5230889,5.0,nM,=,8.301030,CHEMBL5230100,Inhibition of EGFR (unknown origin) using FAM-labeled peptide as substrate incubated for 10 mins...,None
2,17758419,CHEMBL1873475,CHEMBL3888240,0.5,nM,=,9.301030,CHEMBL3886627,In Vitro HotSpot Kinase Assay: IC50s were determined using the in vitro HotSpot kinase assay (pu...,None
3,17758501,CHEMBL1873475,CHEMBL3888240,0.5,nM,=,9.301030,CHEMBL3886627,In Vitro HotSpot Kinase Assay: IC50s were determined using the in vitro HotSpot kinase assay (pu...,None
4,17758440,CHEMBL3647967,CHEMBL3888240,20.6,nM,=,7.686133,CHEMBL3886627,In Vitro HotSpot Kinase Assay: IC50s were determined using the in vitro HotSpot kinase assay (pu...,None
5,17758509,CHEMBL3647967,CHEMBL3888240,20.6,nM,=,7.686133,CHEMBL3886627,In Vitro HotSpot Kinase Assay: IC50s were determined using the in vitro HotSpot kinase assay (pu...,None
6,27916565,CHEMBL5744203,CHEMBL5735011,550.0,nM,=,6.259637,CHEMBL5727616,EGFR Binding Affinity: EGFR binding affinity was determined using a time-resolved fluorescence r...,None
7,27916574,CHEMBL5744203,CHEMBL5735011,550.0,nM,=,6.259637,CHEMBL5727616,EGFR Binding Affinity: EGFR binding affinity was determined using a time-resolved fluorescence r...,None
8,27916583,CHEMBL5744203,CHEMBL5735011,550.0,nM,=,6.259637,CHEMBL5727616,EGFR Binding Affinity: EGFR binding affinity was determined using a time-resolved fluorescence r...,None
9,28162787,CHEMBL5752119,CHEMBL5735965,214.0,nM,=,6.669586,CHEMBL5728067,TR-FRET LanthaScreen Assay: EGFR: TR-FRET LanthaScreen assays were performed by incubating a dil...,None


In [ ]:
# Apply the same quality criteria, but do not exclude
# records solely because of the potential_duplicate flag.

sensitivity_df = candidate_df[
    candidate_df["data_validity_comment"].isna()
    & (candidate_df["standard_value"] > 0)
    & candidate_df["canonical_smiles"].notna()
    & candidate_df["pIC50"].notna()
].copy()

# Identify exact repeated experimental measurements
deduplication_columns = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_value",
    "standard_units",
    "standard_relation",
    "document_chembl_id"
]

before_deduplication = len(sensitivity_df)

sensitivity_df = sensitivity_df.drop_duplicates(
    subset=deduplication_columns,
    keep="first"
).copy()

print("SENSITIVITY DATASET")
print("=" * 55)

print("Records before deduplication:", before_deduplication)
print("Records after deduplication:", len(sensitivity_df))
print("Exact repeated entries removed:",
      before_deduplication - len(sensitivity_df))

print("Unique compounds:",
      sensitivity_df["molecule_chembl_id"].nunique())

print("Unique assays:",
      sensitivity_df["assay_chembl_id"].nunique())

print("\npIC50 statistics:")
display(sensitivity_df["pIC50"].describe().to_frame())

SENSITIVITY DATASET
Records before deduplication: 7939
Records after deduplication: 7907
Exact repeated entries removed: 32
Unique compounds: 6128
Unique assays: 844

pIC50 statistics:


,pIC50
count,7907.000000
mean,7.139292
std,1.359177
min,4.000000
25%,6.142668
50%,7.229148
75%,8.221849
max,11.000000


In [ ]:
comparison = pd.DataFrame({
    "Primary dataset": clean_candidates["pIC50"].describe(),
    "Sensitivity dataset": sensitivity_df["pIC50"].describe()
})

display(comparison)

,Primary dataset,Sensitivity dataset
count,7000.000000,7907.000000
mean,7.118543,7.139292
std,1.357325,1.359177
min,4.000000,4.000000
25%,6.111579,6.142668
50%,7.200659,7.229148
75%,8.200659,8.221849
max,11.000000,11.000000


In [ ]:
# Summarise measurements for each compound

compound_measurements = (
    sensitivity_df
    .groupby("molecule_chembl_id")
    .agg(
        activity_records=("activity_id", "count"),
        unique_assays=("assay_chembl_id", "nunique"),
        unique_pIC50=("pIC50", "nunique"),
        mean_pIC50=("pIC50", "mean"),
        median_pIC50=("pIC50", "median"),
        std_pIC50=("pIC50", "std"),
        min_pIC50=("pIC50", "min"),
        max_pIC50=("pIC50", "max")
    )
    .reset_index()
)

compound_measurements["pIC50_range"] = (
    compound_measurements["max_pIC50"]
    - compound_measurements["min_pIC50"]
)

print("COMPOUND-LEVEL MEASUREMENT AUDIT")
print("=" * 60)

print("Total unique compounds:", len(compound_measurements))

print(
    "Compounds with multiple activity records:",
    (compound_measurements["activity_records"] > 1).sum()
)

print(
    "Compounds measured in multiple assays:",
    (compound_measurements["unique_assays"] > 1).sum()
)

print(
    "Compounds with varying pIC50 values:",
    (compound_measurements["unique_pIC50"] > 1).sum()
)

print("\nACTIVITY RECORD DISTRIBUTION PER COMPOUND")

display(
    compound_measurements["activity_records"]
    .value_counts()
    .sort_index()
    .rename_axis("Number of activity records")
    .to_frame("Number of compounds")
)

print("\nCOMPOUNDS WITH THE LARGEST pIC50 RANGE")

display(
    compound_measurements
    .sort_values("pIC50_range", ascending=False)
    .head(10)
)

COMPOUND-LEVEL MEASUREMENT AUDIT
Total unique compounds: 6128
Compounds with multiple activity records: 1027
Compounds measured in multiple assays: 953
Compounds with varying pIC50 values: 561

ACTIVITY RECORD DISTRIBUTION PER COMPOUND


,Number of compounds
Number of activity records,
1,5101
2,768
3,185
4,31
5,18
6,6
8,4
12,2
13,1



COMPOUNDS WITH THE LARGEST pIC50 RANGE


,molecule_chembl_id,activity_records,unique_assays,unique_pIC50,mean_pIC50,median_pIC50,std_pIC50,min_pIC50,max_pIC50,pIC50_range
5888,CHEMBL607707,12,12,10,7.396479,7.411737,1.341290,4.097000,9.619789,5.522789
3987,CHEMBL553,75,75,46,7.943591,7.692504,1.006165,5.206908,10.537602,5.330694
6108,CHEMBL939,92,92,59,7.957306,7.688375,0.963869,4.809668,10.000000,5.190332
1420,CHEMBL285063,13,12,7,8.310552,8.698970,1.361620,5.455932,10.096910,4.640978
4635,CHEMBL5832255,3,3,3,6.489631,6.982967,2.186402,4.098711,8.387216,4.288506
1434,CHEMBL29197,20,20,12,8.663271,8.336832,1.657077,6.638272,10.602060,3.963788
4410,CHEMBL5786240,4,4,4,7.188017,7.267450,1.621745,5.217169,9.000000,3.782831
5472,CHEMBL5985978,4,4,4,7.690725,7.852107,1.692955,5.660747,9.397940,3.737193
5255,CHEMBL5948958,3,3,3,7.928385,8.698970,1.971245,5.688246,9.397940,3.709694
4611,CHEMBL5828125,3,3,3,7.913182,8.744727,1.942073,5.693789,9.301030,3.607241


In [ ]:
# Analyse measurements at the compound-assay level

compound_assay_audit = (
    sensitivity_df
    .groupby(["molecule_chembl_id", "assay_chembl_id"])
    .agg(
        activity_records=("activity_id", "count"),
        unique_pIC50=("pIC50", "nunique"),
        mean_pIC50=("pIC50", "mean"),
        min_pIC50=("pIC50", "min"),
        max_pIC50=("pIC50", "max")
    )
    .reset_index()
)

compound_assay_audit["pIC50_range"] = (
    compound_assay_audit["max_pIC50"]
    - compound_assay_audit["min_pIC50"]
)

print("WITHIN-ASSAY VARIABILITY")
print("=" * 60)

print(
    "Total compound-assay combinations:",
    len(compound_assay_audit)
)

print(
    "Combinations with repeated measurements:",
    (compound_assay_audit["activity_records"] > 1).sum()
)

print(
    "Combinations with varying pIC50:",
    (compound_assay_audit["unique_pIC50"] > 1).sum()
)

print("\nREPEATED MEASUREMENTS WITHIN THE SAME ASSAY")

display(
    compound_assay_audit[
        compound_assay_audit["activity_records"] > 1
    ]
    .sort_values("pIC50_range", ascending=False)
    .head(15)
)

print("\nDISTRIBUTION OF WITHIN-ASSAY pIC50 RANGES")

display(
    compound_assay_audit.loc[
        compound_assay_audit["activity_records"] > 1,
        "pIC50_range"
    ].describe().to_frame()
)

WITHIN-ASSAY VARIABILITY
Total compound-assay combinations: 7804
Combinations with repeated measurements: 98
Combinations with varying pIC50: 98

REPEATED MEASUREMENTS WITHIN THE SAME ASSAY


,molecule_chembl_id,assay_chembl_id,activity_records,unique_pIC50,mean_pIC50,min_pIC50,max_pIC50,pIC50_range
6530,CHEMBL5939157,CHEMBL5736995,2,2,7.199840,5.399681,9.000000,3.600319
6636,CHEMBL5952854,CHEMBL5730778,2,2,8.489985,6.924453,10.055517,3.131064
6420,CHEMBL5924696,CHEMBL5736995,3,3,7.951705,6.156145,9.000000,2.843855
5588,CHEMBL5805378,CHEMBL5736995,5,5,8.168328,6.397940,9.000000,2.602060
6635,CHEMBL5952854,CHEMBL5730777,2,2,8.819987,7.537602,10.102373,2.564771
6634,CHEMBL5952854,CHEMBL5730776,2,2,7.875034,6.595166,9.154902,2.559736
5449,CHEMBL5779576,CHEMBL5736995,2,2,7.211945,6.025949,8.397940,2.371991
6142,CHEMBL5885640,CHEMBL5736995,2,2,7.403438,6.283997,8.522879,2.238882
5255,CHEMBL5747445,CHEMBL5736995,2,2,6.879538,5.935168,7.823909,1.888741
5742,CHEMBL5826541,CHEMBL5736995,3,3,7.744133,6.612610,8.397940,1.785330



DISTRIBUTION OF WITHIN-ASSAY pIC50 RANGES


,pIC50_range
count,98.000000
mean,0.681230
std,0.774200
min,0.000022
25%,0.096910
50%,0.467880
75%,0.903090
max,3.600319


In [ ]:
# Retrieve the five most variable compound-assay combinations
top_variable_records = sensitivity_df.merge(
    top_variable_pairs[
        ["molecule_chembl_id", "assay_chembl_id", "pIC50_range"]
    ],
    on=["molecule_chembl_id", "assay_chembl_id"],
    how="inner"
)

columns_to_inspect = [
    "activity_id",
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_value",
    "standard_units",
    "pIC50",
    "pIC50_range",
    "assay_type",
    "bao_label",
    "assay_description",
    "document_chembl_id",
    "document_year"
]

display(
    top_variable_records[columns_to_inspect]
    .sort_values(
        ["pIC50_range", "molecule_chembl_id"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

,activity_id,molecule_chembl_id,assay_chembl_id,standard_value,standard_units,pIC50,pIC50_range,assay_type,bao_label,assay_description,document_chembl_id,document_year
0,28404131,CHEMBL5939157,CHEMBL5736995,1.000,nM,9.000000,3.600319,B,single protein format,"High Throughput Biochemical Assay: EGFR activity was measured using KinEASE (Cisbio), a time-res...",CHEMBL5728555,2021.0
1,28406027,CHEMBL5939157,CHEMBL5736995,3984.000,nM,5.399681,3.600319,B,single protein format,"High Throughput Biochemical Assay: EGFR activity was measured using KinEASE (Cisbio), a time-res...",CHEMBL5728555,2021.0
2,26749487,CHEMBL5952854,CHEMBL5730778,0.088,nM,10.055517,3.131064,B,single protein format,TBD: TBD,CHEMBL5725419,2017.0
3,26749901,CHEMBL5952854,CHEMBL5730778,119.000,nM,6.924453,3.131064,B,single protein format,TBD: TBD,CHEMBL5725419,2017.0
4,28403171,CHEMBL5924696,CHEMBL5736995,2.000,nM,8.698970,2.843855,B,single protein format,"High Throughput Biochemical Assay: EGFR activity was measured using KinEASE (Cisbio), a time-res...",CHEMBL5728555,2021.0
5,28403327,CHEMBL5924696,CHEMBL5736995,1.000,nM,9.000000,2.843855,B,single protein format,"High Throughput Biochemical Assay: EGFR activity was measured using KinEASE (Cisbio), a time-res...",CHEMBL5728555,2021.0
6,28403474,CHEMBL5924696,CHEMBL5736995,698.000,nM,6.156145,2.843855,B,single protein format,"High Throughput Biochemical Assay: EGFR activity was measured using KinEASE (Cisbio), a time-res...",CHEMBL5728555,2021.0
7,28403330,CHEMBL5805378,CHEMBL5736995,1.000,nM,9.000000,2.602060,B,single protein format,"High Throughput Biochemical Assay: EGFR activity was measured using KinEASE (Cisbio), a time-res...",CHEMBL5728555,2021.0
8,28403333,CHEMBL5805378,CHEMBL5736995,400.000,nM,6.397940,2.602060,B,single protein format,"High Throughput Biochemical Assay: EGFR activity was measured using KinEASE (Cisbio), a time-res...",CHEMBL5728555,2021.0
9,28403363,CHEMBL5805378,CHEMBL5736995,2.000,nM,8.698970,2.602060,B,single protein format,"High Throughput Biochemical Assay: EGFR activity was measured using KinEASE (Cisbio), a time-res...",CHEMBL5728555,2021.0


In [ ]:
# Select the five most variable compound-assay combinations
# and retrieve their original activity records.

metadata_columns = [
    "activity_id",
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_value",
    "standard_units",
    "standard_relation",
    "pIC50",
    "pchembl_value",
    "activity_comment",
    "standard_text",
    "standard_flag",
    "data_validity_comment",
    "potential_duplicate",
    "assay_type",
    "assay_description",
    "assay_variant_accession",
    "assay_variant_mutation",
    "bao_format",
    "bao_label",
    "document_chembl_id",
    "document_journal",
    "document_year",
    "src_id",
    "record_id"
]

# Keep only columns actually available in the dataset
available_columns = [
    col for col in metadata_columns
    if col in sensitivity_df.columns
]

# Inspect the original records
display(
    top_variable_records[available_columns]
    .sort_values(
        ["molecule_chembl_id", "standard_value"]
    )
    .reset_index(drop=True)
)

,activity_id,molecule_chembl_id,assay_chembl_id,standard_value,standard_units,standard_relation,pIC50,pchembl_value,activity_comment,standard_flag,data_validity_comment,potential_duplicate,assay_type,assay_description,assay_variant_accession,assay_variant_mutation,bao_format,bao_label,document_chembl_id,document_journal,document_year,src_id,record_id
0,28403330,CHEMBL5805378,CHEMBL5736995,1.000,nM,=,9.000000,9.00,1058663,1,None,0,B,"High Throughput Biochemical Assay: EGFR activity was measured using KinEASE (Cisbio), a time-res...",None,None,BAO_0000357,single protein format,CHEMBL5728555,None,2021.0,37,4509570
1,28403363,CHEMBL5805378,CHEMBL5736995,2.000,nM,=,8.698970,8.70,1058674,1,None,0,B,"High Throughput Biochemical Assay: EGFR activity was measured using KinEASE (Cisbio), a time-res...",None,None,BAO_0000357,single protein format,CHEMBL5728555,None,2021.0,37,4509570
2,28403510,CHEMBL5805378,CHEMBL5736995,3.000,nM,=,8.522879,8.52,1058723,1,None,0,B,"High Throughput Biochemical Assay: EGFR activity was measured using KinEASE (Cisbio), a time-res...",None,None,BAO_0000357,single protein format,CHEMBL5728555,None,2021.0,37,4509628
3,28403480,CHEMBL5805378,CHEMBL5736995,6.000,nM,=,8.221849,8.22,1058713,1,None,0,B,"High Throughput Biochemical Assay: EGFR activity was measured using KinEASE (Cisbio), a time-res...",None,None,BAO_0000357,single protein format,CHEMBL5728555,None,2021.0,37,4509618
4,28403333,CHEMBL5805378,CHEMBL5736995,400.000,nM,=,6.397940,6.40,1058664,1,None,0,B,"High Throughput Biochemical Assay: EGFR activity was measured using KinEASE (Cisbio), a time-res...",None,None,BAO_0000357,single protein format,CHEMBL5728555,None,2021.0,37,4509570
5,28403327,CHEMBL5924696,CHEMBL5736995,1.000,nM,=,9.000000,9.00,1058662,1,None,0,B,"High Throughput Biochemical Assay: EGFR activity was measured using KinEASE (Cisbio), a time-res...",None,None,BAO_0000357,single protein format,CHEMBL5728555,None,2021.0,37,4509518
6,28403171,CHEMBL5924696,CHEMBL5736995,2.000,nM,=,8.698970,8.70,1058610,1,None,0,B,"High Throughput Biochemical Assay: EGFR activity was measured using KinEASE (Cisbio), a time-res...",None,None,BAO_0000357,single protein format,CHEMBL5728555,None,2021.0,37,4509518
7,28403474,CHEMBL5924696,CHEMBL5736995,698.000,nM,=,6.156145,6.16,1058711,1,None,0,B,"High Throughput Biochemical Assay: EGFR activity was measured using KinEASE (Cisbio), a time-res...",None,None,BAO_0000357,single protein format,CHEMBL5728555,None,2021.0,37,4509616
8,28404131,CHEMBL5939157,CHEMBL5736995,1.000,nM,=,9.000000,9.00,1058930,1,None,0,B,"High Throughput Biochemical Assay: EGFR activity was measured using KinEASE (Cisbio), a time-res...",None,None,BAO_0000357,single protein format,CHEMBL5728555,None,2021.0,37,4509835
9,28406027,CHEMBL5939157,CHEMBL5736995,3984.000,nM,=,5.399681,5.40,1059562,1,None,0,B,"High Throughput Biochemical Assay: EGFR activity was measured using KinEASE (Cisbio), a time-res...",None,None,BAO_0000357,single protein format,CHEMBL5728555,None,2021.0,37,4509835


In [ ]:
# Assays and publications requiring further investigation

assay_ids = [
    "CHEMBL5736995",
    "CHEMBL5730777",
    "CHEMBL5730778"
]

document_ids = [
    "CHEMBL5728555",
    "CHEMBL5725419"
]

# Retrieve assay metadata
assay_details = []

for assay_id in assay_ids:

    response = requests.get(
        f"{BASE_URL}/assay/{assay_id}.json",
        timeout=30
    )
    response.raise_for_status()

    assay_details.append(response.json())

assay_metadata_df = pd.DataFrame(assay_details)

assay_fields = [
    "assay_chembl_id",
    "assay_type",
    "assay_description",
    "assay_category",
    "assay_test_type",
    "bao_format",
    "target_chembl_id",
    "confidence_score",
    "assay_variant_accession",
    "assay_variant_mutation",
    "document_chembl_id"
]

display(
    assay_metadata_df[
        [col for col in assay_fields if col in assay_metadata_df.columns]
    ].T
)

# Retrieve publication metadata
document_details = []

for document_id in document_ids:

    response = requests.get(
        f"{BASE_URL}/document/{document_id}.json",
        timeout=30
    )
    response.raise_for_status()

    document_details.append(response.json())

document_metadata_df = pd.DataFrame(document_details)

document_fields = [
    "document_chembl_id",
    "title",
    "journal",
    "year",
    "doi",
    "pubmed_id",
    "authors"
]

display(
    document_metadata_df[
        [col for col in document_fields if col in document_metadata_df.columns]
    ].T
)

,0,1,2
assay_chembl_id,CHEMBL5736995,CHEMBL5730777,CHEMBL5730778
assay_type,B,B,B
assay_category,None,None,None
assay_test_type,None,None,None
bao_format,BAO_0000357,BAO_0000357,BAO_0000357
target_chembl_id,CHEMBL203,CHEMBL203,CHEMBL203
confidence_score,9,8,8
document_chembl_id,CHEMBL5728555,CHEMBL5725419,CHEMBL5725419


,0,1
document_chembl_id,CHEMBL5728555,CHEMBL5725419
title,Cot modulators and methods of use thereof,Fused tricyclic amide compounds as multiple kinase inhibitors
journal,None,None
year,2021,2017
doi,None,None
pubmed_id,None,None
authors,None,None


In [ ]:
print("Columns containing 'confidence':")

print([
    col for col in df.columns
    if "confidence" in col.lower()
])

print("\nAvailable confidence-related fields:")
print([
    col for col in assay_metadata_df.columns
    if "confidence" in col.lower()
])

Columns containing 'confidence':
[]

Available confidence-related fields:
['confidence_description', 'confidence_score']


In [ ]:
print("Activity DataFrame columns:")
print(df.columns.tolist())

print("\nAssay metadata columns:")
print(assay_metadata_df.columns.tolist())

Activity DataFrame columns:
['action_type', 'activity_comment', 'activity_id', 'activity_properties', 'assay_chembl_id', 'assay_description', 'assay_type', 'assay_variant_accession', 'assay_variant_mutation', 'bao_endpoint', 'bao_format', 'bao_label', 'canonical_smiles', 'data_validity_comment', 'data_validity_description', 'document_chembl_id', 'document_journal', 'document_year', 'ligand_efficiency', 'modality', 'molecule_chembl_id', 'molecule_pref_name', 'parent_molecule_chembl_id', 'pchembl_value', 'potential_duplicate', 'qudt_units', 'record_id', 'relation', 'src_id', 'standard_flag', 'standard_relation', 'standard_text_value', 'standard_type', 'standard_units', 'standard_upper_value', 'standard_value', 'target_chembl_id', 'target_organism', 'target_pref_name', 'target_tax_id', 'text_value', 'toid', 'type', 'units', 'uo_units', 'upper_value', 'value', 'pIC50']

Assay metadata columns:
['aidx', 'assay_category', 'assay_cell_type', 'assay_chembl_id', 'assay_classifications', 'assay_

In [ ]:
print(
    assay_metadata_df[
        ["assay_chembl_id", "confidence_score"]
    ].to_string(index=False)
)

assay_chembl_id  confidence_score
  CHEMBL5736995                 9
  CHEMBL5730777                 8
  CHEMBL5730778                 8


In [ ]:
assay_check = (
    df[df["assay_chembl_id"].isin(assay_ids)]
    .groupby("assay_chembl_id")
    .agg(
        activity_records=("activity_id", "count"),
        unique_compounds=("molecule_chembl_id", "nunique"),
        document=("document_chembl_id", "first")
    )
    .reset_index()
)

assay_check = assay_check.merge(
    assay_metadata_df[
        ["assay_chembl_id", "confidence_score"]
    ],
    on="assay_chembl_id",
    how="left"
)

display(assay_check)

,assay_chembl_id,activity_records,unique_compounds,document,confidence_score
0,CHEMBL5730777,83,82,CHEMBL5725419,8
1,CHEMBL5730778,82,80,CHEMBL5725419,8
2,CHEMBL5736995,975,911,CHEMBL5728555,9


In [ ]:
assay_url = f"{BASE_URL}/assay.json"

assay_params = {
    "target_chembl_id": TARGET_ID,
    "limit": 1000
}

all_assay_metadata = fetch_all_chembl_records(
    assay_url,
    assay_params
)

PAGINATION REPORT
Expected records: 5966
Downloaded records: 0
Records remaining: 5966


In [ ]:
import time
import requests

def fetch_assays_in_batches(target_id, batch_size=250):

    session = requests.Session()
    records = []
    offset = 0
    total_expected = None

    while True:

        params = {
            "target_chembl_id": target_id,
            "limit": batch_size,
            "offset": offset
        }

        for attempt in range(4):

            try:
                response = session.get(
                    f"{BASE_URL}/assay.json",
                    params=params,
                    timeout=90
                )

                response.raise_for_status()
                data = response.json()
                break

            except requests.exceptions.RequestException as error:

                print(
                    f"Batch offset {offset}, "
                    f"attempt {attempt + 1}: {error}"
                )

                if attempt == 3:
                    raise

                time.sleep(3 * (attempt + 1))

        batch = data.get("assays", [])

        if total_expected is None:
            total_expected = data["page_meta"]["total_count"]

        records.extend(batch)

        print(
            f"Retrieved {len(records)} / {total_expected}"
        )

        if not batch or len(records) >= total_expected:
            break

        offset += batch_size
        time.sleep(0.5)

    return records

In [ ]:
all_assay_metadata = fetch_assays_in_batches(
    TARGET_ID,
    batch_size=250
)

all_assay_metadata_df = pd.DataFrame(
    all_assay_metadata
)

print("\nFinal assay dataset shape:",
      all_assay_metadata_df.shape)

Retrieved 250 / 5966
Retrieved 500 / 5966
Retrieved 750 / 5966
Retrieved 1000 / 5966
Retrieved 1250 / 5966
Retrieved 1500 / 5966
Retrieved 1750 / 5966
Retrieved 2000 / 5966
Retrieved 2250 / 5966
Retrieved 2500 / 5966
Retrieved 2750 / 5966
Retrieved 3000 / 5966
Retrieved 3250 / 5966
Retrieved 3500 / 5966
Retrieved 3750 / 5966
Retrieved 4000 / 5966
Retrieved 4250 / 5966
Retrieved 4500 / 5966
Retrieved 4750 / 5966
Retrieved 5000 / 5966
Retrieved 5250 / 5966
Retrieved 5500 / 5966
Retrieved 5750 / 5966
Retrieved 5966 / 5966

Final assay dataset shape: (5966, 29)


In [ ]:
display(
    all_assay_metadata_df["confidence_score"]
    .value_counts(dropna=False)
    .sort_index()
)

,count
confidence_score,
8,729
9,5237


In [ ]:
# Keep the assay ID and confidence score from the complete metadata
assay_confidence = (
    all_assay_metadata_df[
        ["assay_chembl_id", "confidence_score"]
    ]
    .drop_duplicates(subset="assay_chembl_id")
)

# Attach assay confidence to the original activity dataset
df_audited = df.merge(
    assay_confidence,
    on="assay_chembl_id",
    how="left",
    validate="many_to_one"
)

print("ACTIVITY-LEVEL CONFIDENCE AUDIT")
print("=" * 50)

print("\nActivity records by assay confidence:")
print(
    df_audited["confidence_score"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nUnique assays represented in activity data:")
print(df_audited["assay_chembl_id"].nunique())

print("\nUnique assays by confidence:")
print(
    df_audited.groupby("confidence_score")["assay_chembl_id"]
    .nunique()
)

ACTIVITY-LEVEL CONFIDENCE AUDIT

Activity records by assay confidence:
confidence_score
8     4838
9    14808
Name: count, dtype: int64

Unique assays represented in activity data:
2044

Unique assays by confidence:
confidence_score
8     349
9    1695
Name: assay_chembl_id, dtype: int64


In [ ]:
confidence_8_records = df_audited[
    df_audited["confidence_score"] == 8
]

print("Confidence-8 activity records:",
      len(confidence_8_records))

print("Unique confidence-8 assays:",
      confidence_8_records["assay_chembl_id"].nunique())

print("Unique confidence-8 compounds:",
      confidence_8_records["molecule_chembl_id"].nunique())

display(
    confidence_8_records[
        [
            "assay_chembl_id",
            "molecule_chembl_id",
            "standard_value",
            "standard_units",
            "document_chembl_id"
        ]
    ].head(10)
)

Confidence-8 activity records: 4838
Unique confidence-8 assays: 349
Unique confidence-8 compounds: 3267


,assay_chembl_id,molecule_chembl_id,standard_value,standard_units,document_chembl_id
0,CHEMBL674637,CHEMBL68920,41.0,nM,CHEMBL1134862
3,CHEMBL674637,CHEMBL69960,170.0,nM,CHEMBL1134862
6,CHEMBL677833,CHEMBL137635,9300.0,nM,CHEMBL1145114
7,CHEMBL674643,CHEMBL306988,500000.0,nM,CHEMBL1124610
8,CHEMBL674643,CHEMBL66879,3000000.0,nM,CHEMBL1124610
9,CHEMBL674643,CHEMBL77085,96000.0,nM,CHEMBL1124610
10,CHEMBL674637,CHEMBL443268,5310.0,nM,CHEMBL1134862
11,CHEMBL674643,CHEMBL76979,264000.0,nM,CHEMBL1124610
12,CHEMBL674643,CHEMBL76589,125.0,nM,CHEMBL1124610
13,CHEMBL674643,CHEMBL76904,35000.0,nM,CHEMBL1124610


In [ ]:
# Recreate the preliminary mutation annotation
confidence_audit_df = df_audited.copy()

mutation_pattern = (
    r"mutant|mutation|L858R|T790M|C797S|"
    r"exon 19|deletion|L834R"
)

confidence_audit_df["mutation_flag"] = (
    confidence_audit_df["assay_variant_mutation"].notna()
    | confidence_audit_df["assay_description"].str.contains(
        mutation_pattern,
        case=False,
        na=False
    )
)

# Apply the original assay-selection criteria
confidence9_candidates = confidence_audit_df[
    (confidence_audit_df["confidence_score"] == 9)
    & (confidence_audit_df["assay_type"] == "B")
    & (confidence_audit_df["bao_label"] == "single protein format")
    & (~confidence_audit_df["mutation_flag"])
].copy()

# Apply the existing strict data-quality criteria
confidence9_clean = confidence9_candidates[
    confidence9_candidates["data_validity_comment"].isna()
    & (confidence9_candidates["potential_duplicate"] == 0)
    & (confidence9_candidates["standard_value"] > 0)
    & confidence9_candidates["canonical_smiles"].notna()
    & confidence9_candidates["pIC50"].notna()
].copy()

# Summary
print("CONFIDENCE-9 DATASET RECONSTRUCTION")
print("=" * 55)

print("\nPreliminary candidate records:",
      len(confidence9_candidates))

print("Unique candidate compounds:",
      confidence9_candidates["molecule_chembl_id"].nunique())

print("Unique candidate assays:",
      confidence9_candidates["assay_chembl_id"].nunique())

print("\nStrict clean records:",
      len(confidence9_clean))

print("Unique clean compounds:",
      confidence9_clean["molecule_chembl_id"].nunique())

print("Unique clean assays:",
      confidence9_clean["assay_chembl_id"].nunique())

print("\nClean dataset pIC50 statistics:")
display(confidence9_clean["pIC50"].describe())

CONFIDENCE-9 DATASET RECONSTRUCTION

Preliminary candidate records: 5406
Unique candidate compounds: 4386
Unique candidate assays: 627

Strict clean records: 4696
Unique clean compounds: 4181
Unique clean assays: 538

Clean dataset pIC50 statistics:


,pIC50
count,4696.000000
mean,7.255847
std,1.266323
min,4.000000
25%,6.276093
50%,7.376751
75%,8.267606
max,11.000000


In [ ]:
comparison = pd.DataFrame({
    "Dataset": [
        "Previous primary clean dataset",
        "Reconstructed confidence-9 clean dataset"
    ],
    "Activity records": [
        len(clean_candidates),
        len(confidence9_clean)
    ],
    "Unique compounds": [
        clean_candidates["molecule_chembl_id"].nunique(),
        confidence9_clean["molecule_chembl_id"].nunique()
    ],
    "Unique assays": [
        clean_candidates["assay_chembl_id"].nunique(),
        confidence9_clean["assay_chembl_id"].nunique()
    ],
    "Mean pIC50": [
        clean_candidates["pIC50"].mean(),
        confidence9_clean["pIC50"].mean()
    ],
    "Median pIC50": [
        clean_candidates["pIC50"].median(),
        confidence9_clean["pIC50"].median()
    ]
})

display(comparison)

,Dataset,Activity records,Unique compounds,Unique assays,Mean pIC50,Median pIC50
0,Previous primary clean dataset,7000,5938,752,7.118543,7.200659
1,Reconstructed confidence-9 clean dataset,4696,4181,538,7.255847,7.376751


In [ ]:
test_pages = []

for offset in [0, 1000]:

    test_params = {
        "target_chembl_id": TARGET_ID,
        "standard_type": "IC50",
        "standard_relation": "=",
        "standard_units": "nM",
        "confidence_score": 9,
        "limit": 1000,
        "offset": offset
    }

    response = requests.get(
        activity_url,
        params=test_params,
        timeout=90
    )

    response.raise_for_status()

    page_data = response.json()

    page_df = pd.DataFrame(
        page_data.get("activities", [])
    )

    page_df["requested_offset"] = offset

    test_pages.append(page_df)

test_df = pd.concat(test_pages, ignore_index=True)

test_audit = test_df.merge(
    assay_confidence,
    on="assay_chembl_id",
    how="left"
)

print("TEST RECORDS:", len(test_audit))

print("\nAssay confidence distribution by requested page:")

display(
    test_audit.groupby(
        ["requested_offset", "confidence_score"]
    ).size().unstack(fill_value=0)
)

TEST RECORDS: 2000

Assay confidence distribution by requested page:


confidence_score,8,9
requested_offset,,
0,851,149
1000,749,251


In [ ]:
print("ACTIVITY RESPONSE PAGE METADATA")
print("=" * 45)

print(page_data["page_meta"])

print("\nACTIVITY RECORD CONFIDENCE FIELD")

sample_record = page_data["activities"][0]

print("Available fields:")
print(sample_record.keys())

print("\nConfidence score in activity record:")
print(sample_record.get("confidence_score"))

print("\nAssay ID:")
print(sample_record.get("assay_chembl_id"))

ACTIVITY RESPONSE PAGE METADATA
{'limit': 1000, 'next': '/chembl/api/data/activity.json?limit=1000&offset=2000&target_chembl_id=CHEMBL203&standard_type=IC50&standard_relation=%3D&standard_units=nM&confidence_score=9', 'offset': 1000, 'previous': '/chembl/api/data/activity.json?limit=1000&offset=0&target_chembl_id=CHEMBL203&standard_type=IC50&standard_relation=%3D&standard_units=nM&confidence_score=9', 'total_count': 19646}

ACTIVITY RECORD CONFIDENCE FIELD
Available fields:
dict_keys(['action_type', 'activity_comment', 'activity_id', 'activity_properties', 'assay_chembl_id', 'assay_description', 'assay_type', 'assay_variant_accession', 'assay_variant_mutation', 'bao_endpoint', 'bao_format', 'bao_label', 'canonical_smiles', 'data_validity_comment', 'data_validity_description', 'document_chembl_id', 'document_journal', 'document_year', 'ligand_efficiency', 'modality', 'molecule_chembl_id', 'molecule_pref_name', 'parent_molecule_chembl_id', 'pchembl_value', 'potential_duplicate', 'qudt_un

In [ ]:
print(response.url)

https://www.ebi.ac.uk/chembl/api/data/activity.json?target_chembl_id=CHEMBL203&standard_type=IC50&standard_relation=%3D&standard_units=nM&confidence_score=9&limit=1000&offset=1000


In [ ]:
test_params_8 = {
    "target_chembl_id": TARGET_ID,
    "standard_type": "IC50",
    "standard_relation": "=",
    "standard_units": "nM",
    "confidence_score": 8,
    "limit": 1000
}

response_8 = requests.get(
    activity_url,
    params=test_params_8,
    timeout=90
)

response_8.raise_for_status()

data_8 = response_8.json()

print("CONFIDENCE PARAMETER TEST")
print("=" * 45)

print("Requested confidence: 8")
print("API total count:", data_8["page_meta"]["total_count"])

print("\nRequested confidence: 9")
print("API total count:", data["page_meta"]["total_count"])

CONFIDENCE PARAMETER TEST
Requested confidence: 8
API total count: 19646

Requested confidence: 9
API total count: 19646


In [ ]:
confidence_filter_results = []

for score in [8, 9]:

    test_params = {
        "target_chembl_id": TARGET_ID,
        "standard_type": "IC50",
        "standard_relation": "=",
        "standard_units": "nM",
        "assay__confidence_score": score,
        "limit": 1000
    }

    response = requests.get(
        activity_url,
        params=test_params,
        timeout=90
    )

    response.raise_for_status()

    page_data = response.json()

    page_df = pd.DataFrame(
        page_data.get("activities", [])
    )

    page_df = page_df.merge(
        assay_confidence,
        on="assay_chembl_id",
        how="left"
    )

    confidence_filter_results.append({
        "requested_confidence": score,
        "api_total_count": page_data["page_meta"]["total_count"],
        "records_on_first_page": len(page_df),
        "confidence_8_records": (
            page_df["confidence_score"] == 8
        ).sum(),
        "confidence_9_records": (
            page_df["confidence_score"] == 9
        ).sum()
    })

display(pd.DataFrame(confidence_filter_results))

,requested_confidence,api_total_count,records_on_first_page,confidence_8_records,confidence_9_records
0,8,19646,1000,851,149
1,9,19646,1000,851,149


In [ ]:
print("CONFIDENCE SCORE VALIDATION")
print("=" * 45)

print(
    confidence9_clean["confidence_score"]
    .value_counts(dropna=False)
)

print("\nUnique confidence scores:",
      confidence9_clean["confidence_score"].unique())

assert confidence9_clean["confidence_score"].eq(9).all()

print("\nSUCCESS: All records belong to confidence-9 assays.")

CONFIDENCE SCORE VALIDATION
confidence_score
9    4696
Name: count, dtype: int64

Unique confidence scores: [9]

SUCCESS: All records belong to confidence-9 assays.


In [ ]:
assay_profile = (
    confidence9_clean
    .groupby("assay_chembl_id")
    .agg(
        activity_records=("activity_id", "count"),
        unique_compounds=("molecule_chembl_id", "nunique"),
        mean_pIC50=("pIC50", "mean"),
        median_pIC50=("pIC50", "median"),
        std_pIC50=("pIC50", "std"),
        min_pIC50=("pIC50", "min"),
        max_pIC50=("pIC50", "max"),
        assay_description=("assay_description", "first"),
        document_chembl_id=("document_chembl_id", "first"),
        document_year=("document_year", "first")
    )
    .reset_index()
    .sort_values("activity_records", ascending=False)
)

print("ASSAY-LEVEL DATASET PROFILE")
print("=" * 50)

print("Total unique assays:", len(assay_profile))
print("Total activity records:", assay_profile["activity_records"].sum())
print("Median records per assay:", assay_profile["activity_records"].median())
print("Maximum records in one assay:", assay_profile["activity_records"].max())

print("\nTOP 20 ASSAYS BY ACTIVITY RECORD COUNT")

display(assay_profile.head(20))

ASSAY-LEVEL DATASET PROFILE
Total unique assays: 538
Total activity records: 4696
Median records per assay: 2.0
Maximum records in one assay: 966

TOP 20 ASSAYS BY ACTIVITY RECORD COUNT


,assay_chembl_id,activity_records,unique_compounds,mean_pIC50,median_pIC50,std_pIC50,min_pIC50,max_pIC50,assay_description,document_chembl_id,document_year
432,CHEMBL5736995,966,902,8.144406,8.301030,0.617478,5.015743,9.000000,"High Throughput Biochemical Assay: EGFR activity was measured using KinEASE (Cisbio), a time-res...",CHEMBL5728555,2021.0
76,CHEMBL3705421,173,170,7.356859,7.443697,0.669994,5.000000,8.698970,Tyrosine Kinase Assay: EGFR1 (HER-1) enzyme assay was carried out by using Tyrosine Kinase Assay...,CHEMBL3638799,2014.0
406,CHEMBL5729820,173,170,7.356859,7.443697,0.669994,5.000000,8.698970,Tyrosine Kinase Assay: EGFR1 (HER-1) enzyme assay was carried out by using Tyrosine Kinase Assay...,CHEMBL3638799,2014.0
534,CHEMBL944276,106,106,8.088191,8.569925,1.399316,4.867420,9.799971,Inhibition of human recombinant EGFR,CHEMBL1140899,2009.0
415,CHEMBL5731823,94,94,5.685169,5.259637,0.538604,5.259637,7.259637,HotSpot kinase Assay: The degree of inhibition of a panel of kinases is determined using the in ...,CHEMBL5725990,2017.0
411,CHEMBL5730776,85,83,8.107110,8.522879,1.025270,5.462433,9.366532,TBD: TBD,CHEMBL5725419,2017.0
422,CHEMBL5734361,82,77,5.857198,6.259637,0.625779,5.259637,7.259637,EGFR Binding Affinity: EGFR binding affinity was determined using a time-resolved fluorescence r...,CHEMBL5727286,2019.0
414,CHEMBL5731481,60,60,7.341595,7.442659,0.539202,5.887730,8.886057,EGFR Inhibitory Activity: With regard to the setting of the conditions for a method for measurin...,CHEMBL5725787,2017.0
71,CHEMBL3625155,54,54,7.755031,8.091515,0.833466,6.241391,9.045757,Inhibition of EGFR (unknown origin) by ELISA method,CHEMBL3621102,2015.0
435,CHEMBL5737421,50,48,5.579556,5.641591,0.307133,5.045227,5.999132,Kinase Inhibition Assay: EGFR: The working concentration of each component in 10 μl wild-type EG...,CHEMBL5728771,2021.0


In [ ]:
# 1. Assay concentration analysis

assay_counts = (
    confidence9_clean
    .groupby("assay_chembl_id")
    .size()
    .sort_values(ascending=False)
)

total_records = len(confidence9_clean)

print("ASSAY CONCENTRATION")
print("=" * 50)

for n in [1, 3, 5, 10, 20]:
    records = assay_counts.head(n).sum()
    print(
        f"Top {n:2} assays: {records:4} records "
        f"({records / total_records * 100:.2f}%)"
    )

print("\nAssay count distribution:")
print(
    assay_counts.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90]
    )
)

print("\nNumber of assays by record count:")
print(
    assay_counts.value_counts()
    .sort_index()
    .head(20)
)


# 2. Investigate the two matching assay IDs

assay_a = "CHEMBL3705421"
assay_b = "CHEMBL5729820"

records_a = confidence9_clean[
    confidence9_clean["assay_chembl_id"] == assay_a
]

records_b = confidence9_clean[
    confidence9_clean["assay_chembl_id"] == assay_b
]

compounds_a = set(records_a["molecule_chembl_id"])
compounds_b = set(records_b["molecule_chembl_id"])

print("\n\nASSAY PAIR INVESTIGATION")
print("=" * 50)

print("Assay A:", assay_a)
print("Assay B:", assay_b)

print("Records in A:", len(records_a))
print("Records in B:", len(records_b))

print("Compounds in both:", len(compounds_a & compounds_b))
print("Compounds only in A:", len(compounds_a - compounds_b))
print("Compounds only in B:", len(compounds_b - compounds_a))

# Compare compound-level IC50 measurements

comparison = records_a[
    ["molecule_chembl_id", "standard_value", "pIC50"]
].merge(
    records_b[
        ["molecule_chembl_id", "standard_value", "pIC50"]
    ],
    on="molecule_chembl_id",
    how="inner",
    suffixes=("_A", "_B")
)

print("\nShared compound measurement comparison:")
print("Shared measurement rows:", len(comparison))

print(
    "Identical IC50 values:",
    (
        comparison["standard_value_A"]
        == comparison["standard_value_B"]
    ).sum()
)

print(
    "Identical pIC50 values:",
    np.isclose(
        comparison["pIC50_A"],
        comparison["pIC50_B"]
    ).sum()
)

display(comparison.head(10))

ASSAY CONCENTRATION
Top  1 assays:  966 records (20.57%)
Top  3 assays: 1312 records (27.94%)
Top  5 assays: 1512 records (32.20%)
Top 10 assays: 1843 records (39.25%)
Top 20 assays: 2241 records (47.72%)

Assay count distribution:
count    538.000000
mean       8.728625
std       44.044551
min        1.000000
25%        1.000000
50%        2.000000
75%        7.000000
90%       17.300000
max      966.000000
dtype: float64

Number of assays by record count:
1     223
2      69
3      44
4      28
5      17
6      15
7      20
8      13
9      13
10     12
11      3
12      5
13     11
14      5
15      1
16      2
17      3
18      2
19      3
20      1
Name: count, dtype: int64


ASSAY PAIR INVESTIGATION
Assay A: CHEMBL3705421
Assay B: CHEMBL5729820
Records in A: 173
Records in B: 173
Compounds in both: 170
Compounds only in A: 0
Compounds only in B: 0

Shared compound measurement comparison:
Shared measurement rows: 179
Identical IC50 values: 173
Identical pIC50 values: 173


,molecule_chembl_id,standard_value_A,pIC50_A,standard_value_B,pIC50_B
0,CHEMBL3676392,956.0,6.019542,956.0,6.019542
1,CHEMBL3676391,246.0,6.609065,246.0,6.609065
2,CHEMBL3676393,172.0,6.764472,172.0,6.764472
3,CHEMBL3676394,149.0,6.826814,149.0,6.826814
4,CHEMBL3671475,3.0,8.522879,3.0,8.522879
5,CHEMBL3671476,47.0,7.327902,47.0,7.327902
6,CHEMBL3671477,26.0,7.585027,26.0,7.585027
7,CHEMBL3671478,15.0,7.823909,15.0,7.823909
8,CHEMBL3671479,3.0,8.522879,3.0,8.522879
9,CHEMBL3671480,56.0,7.251812,56.0,7.251812


In [ ]:
# 1. Compare assay-level source metadata

assay_pair_metadata = all_assay_metadata_df[
    all_assay_metadata_df["assay_chembl_id"].isin(
        [assay_a, assay_b]
    )
][[
    "assay_chembl_id",
    "assay_type",
    "confidence_score",
    "bao_format",
    "bao_label",
    "description",
    "document_chembl_id",
    "src_id",
    "src_assay_id",
    "target_chembl_id"
]]

print("ASSAY SOURCE METADATA")
print("=" * 60)
display(assay_pair_metadata.T)


# 2. Compare measurement frequencies by compound and IC50

def measurement_frequency(data):

    return (
        data.groupby(
            ["molecule_chembl_id", "standard_value"]
        )
        .size()
        .reset_index(name="record_count")
    )

freq_a = measurement_frequency(records_a)
freq_b = measurement_frequency(records_b)

measurement_comparison = freq_a.merge(
    freq_b,
    on=["molecule_chembl_id", "standard_value"],
    how="outer",
    suffixes=("_A", "_B")
).fillna(0)

measurement_comparison[
    ["record_count_A", "record_count_B"]
] = measurement_comparison[
    ["record_count_A", "record_count_B"]
].astype(int)

print("\nMEASUREMENT FREQUENCY COMPARISON")
print("=" * 60)

print(
    "Matching compound-IC50 combinations:",
    (
        measurement_comparison["record_count_A"]
        == measurement_comparison["record_count_B"]
    ).sum()
)

print(
    "Different combinations:",
    (
        measurement_comparison["record_count_A"]
        != measurement_comparison["record_count_B"]
    ).sum()
)

display(
    measurement_comparison[
        measurement_comparison["record_count_A"]
        != measurement_comparison["record_count_B"]
    ]
)


# 3. Compare activity record source identifiers

source_columns = [
    "molecule_chembl_id",
    "activity_id",
    "record_id",
    "src_id",
    "document_chembl_id",
    "standard_value"
]

print("\nSOURCE RECORDS — ASSAY A")
display(records_a[source_columns].head(10))

print("\nSOURCE RECORDS — ASSAY B")
display(records_b[source_columns].head(10))

ASSAY SOURCE METADATA


,1714,5613
assay_chembl_id,CHEMBL3705421,CHEMBL5729820
assay_type,B,B
confidence_score,9,9
bao_format,BAO_0000357,BAO_0000357
bao_label,single protein format,single protein format
description,Tyrosine Kinase Assay: EGFR1 (HER-1) enzyme assay was carried out by using Tyrosine Kinase Assay...,Tyrosine Kinase Assay: EGFR1 (HER-1) enzyme assay was carried out by using Tyrosine Kinase Assay...
document_chembl_id,CHEMBL3638799,CHEMBL3638799
src_id,37,37
src_assay_id,6500_2_pol_4952,6500_2_pol_520
target_chembl_id,CHEMBL203,CHEMBL203



MEASUREMENT FREQUENCY COMPARISON
Matching compound-IC50 combinations: 173
Different combinations: 0


,molecule_chembl_id,standard_value,record_count_A,record_count_B



SOURCE RECORDS — ASSAY A


,molecule_chembl_id,activity_id,record_id,src_id,document_chembl_id,standard_value
7849,CHEMBL3676392,17601741,2552133,37,CHEMBL3638799,956.0
7850,CHEMBL3676391,17601742,2552132,37,CHEMBL3638799,246.0
7851,CHEMBL3676393,17601745,2552134,37,CHEMBL3638799,172.0
7852,CHEMBL3676394,17601747,2552135,37,CHEMBL3638799,149.0
7874,CHEMBL3671475,17649031,2546829,37,CHEMBL3638799,3.0
7875,CHEMBL3671476,17649032,2546830,37,CHEMBL3638799,47.0
7876,CHEMBL3671477,17649033,2546831,37,CHEMBL3638799,26.0
7877,CHEMBL3671478,17649034,2546832,37,CHEMBL3638799,15.0
7878,CHEMBL3671479,17649035,2546833,37,CHEMBL3638799,3.0
7879,CHEMBL3671480,17649036,2546834,37,CHEMBL3638799,56.0



SOURCE RECORDS — ASSAY B


,molecule_chembl_id,activity_id,record_id,src_id,document_chembl_id,standard_value
14090,CHEMBL3676392,26575867,2552133,37,CHEMBL3638799,956.0
14091,CHEMBL3676391,26575873,2552132,37,CHEMBL3638799,246.0
14092,CHEMBL3676393,26575876,2552134,37,CHEMBL3638799,172.0
14093,CHEMBL3676394,26575879,2552135,37,CHEMBL3638799,149.0
14122,CHEMBL3671475,26587352,2546829,37,CHEMBL3638799,3.0
14123,CHEMBL3671476,26587355,2546830,37,CHEMBL3638799,47.0
14124,CHEMBL3671477,26587358,2546831,37,CHEMBL3638799,26.0
14125,CHEMBL3671478,26587361,2546832,37,CHEMBL3638799,15.0
14126,CHEMBL3671479,26587364,2546833,37,CHEMBL3638799,3.0
14127,CHEMBL3671480,26587367,2546834,37,CHEMBL3638799,56.0


In [ ]:
# Create a separate dataset for assay-level deduplication

assay_deduplicated = confidence9_clean.copy()

duplicate_assay_id = "CHEMBL5729820"

before_count = len(assay_deduplicated)

assay_deduplicated = assay_deduplicated[
    assay_deduplicated["assay_chembl_id"] != duplicate_assay_id
].copy()

removed_count = before_count - len(assay_deduplicated)

print("ASSAY-LEVEL DUPLICATE REMOVAL")
print("=" * 50)

print("Original records:", before_count)
print("Records removed:", removed_count)
print("Remaining records:", len(assay_deduplicated))

print(
    "\nRemaining records from retained assay:",
    (
        assay_deduplicated["assay_chembl_id"]
        == "CHEMBL3705421"
    ).sum()
)

print(
    "Records from excluded assay:",
    (
        assay_deduplicated["assay_chembl_id"]
        == duplicate_assay_id
    ).sum()
)

ASSAY-LEVEL DUPLICATE REMOVAL
Original records: 4696
Records removed: 173
Remaining records: 4523

Remaining records from retained assay: 173
Records from excluded assay: 0


### Assay-Level Duplicate Investigation

Two assay IDs, CHEMBL3705421 and CHEMBL5729820, were identified as duplicate representations of the same source measurements.

Both records were associated with publication CHEMBL3638799 and source ID 37. Their 173 compound–IC50 combinations had identical measurements and matching source record IDs.

To prevent duplicate experimental entries from receiving additional weight during model training, CHEMBL5729820 was excluded from the working dataset while CHEMBL3705421 was retained.

The original activity records and assay metadata were preserved for traceability.

In [ ]:
from collections import defaultdict

assay_signatures = defaultdict(list)

for assay_id, group in assay_deduplicated.groupby("assay_chembl_id"):

    measurements = tuple(sorted(
        zip(
            group["molecule_chembl_id"],
            group["standard_value"],
            group["standard_units"],
            group["standard_relation"]
        )
    ))

    assay_signatures[measurements].append(assay_id)


# Identify identical measurement sets across different assay IDs

duplicate_assay_groups = [
    assay_ids
    for assay_ids in assay_signatures.values()
    if len(assay_ids) > 1
]

print("DUPLICATE ASSAY SIGNATURE AUDIT")
print("=" * 55)

print("Total unique assays:",
      assay_deduplicated["assay_chembl_id"].nunique())

print("Groups with identical measurement sets:",
      len(duplicate_assay_groups))

for i, assay_ids in enumerate(duplicate_assay_groups, 1):
    print(f"\nPotential duplicate group {i}:")
    print("Assay IDs:", assay_ids)

DUPLICATE ASSAY SIGNATURE AUDIT
Total unique assays: 537
Groups with identical measurement sets: 4

Potential duplicate group 1:
Assay IDs: ['CHEMBL3705330', 'CHEMBL5729840']

Potential duplicate group 2:
Assay IDs: ['CHEMBL3705522', 'CHEMBL5729936']

Potential duplicate group 3:
Assay IDs: ['CHEMBL3705672', 'CHEMBL5729933']

Potential duplicate group 4:
Assay IDs: ['CHEMBL3705968', 'CHEMBL5729875']


In [ ]:
# Inspect source metadata and measurement overlap
# for all four potential duplicate assay pairs.

for i, assay_ids in enumerate(duplicate_assay_groups, 1):

    print("\n" + "=" * 75)
    print(f"POTENTIAL DUPLICATE GROUP {i}")
    print("=" * 75)

    # Assay metadata
    metadata = all_assay_metadata_df[
        all_assay_metadata_df["assay_chembl_id"].isin(assay_ids)
    ]

    metadata_columns = [
        "assay_chembl_id",
        "assay_type",
        "confidence_score",
        "bao_label",
        "description",
        "document_chembl_id",
        "src_id",
        "src_assay_id",
        "target_chembl_id"
    ]

    display(
        metadata[
            [c for c in metadata_columns if c in metadata.columns]
        ].T
    )

    # Activity records
    for assay_id in assay_ids:

        subset = assay_deduplicated[
            assay_deduplicated["assay_chembl_id"] == assay_id
        ]

        print(f"\nAssay: {assay_id}")
        print("Activity records:", len(subset))
        print("Unique compounds:",
              subset["molecule_chembl_id"].nunique())
        print("Publication:",
              subset["document_chembl_id"].unique())

        print("Source IDs:", subset["src_id"].unique())
        print("Source record count:", subset["record_id"].nunique())


POTENTIAL DUPLICATE GROUP 1


,1707,5614
assay_chembl_id,CHEMBL3705330,CHEMBL5729840
assay_type,B,B
confidence_score,9,9
bao_label,single protein format,single protein format
description,Inhibition Assay: EGFR inhibitory activity of compounds of the present invention is quantified e...,Inhibition Assay: EGFR inhibitory activity of compounds of the present invention is quantified e...
document_chembl_id,CHEMBL3638553,CHEMBL3638553
src_id,37,37
src_assay_id,5931_1_pol_4952,5931_1_pol_520
target_chembl_id,CHEMBL203,CHEMBL203



Assay: CHEMBL3705330
Activity records: 21
Unique compounds: 21
Publication: ['CHEMBL3638553']
Source IDs: [37]
Source record count: 21

Assay: CHEMBL5729840
Activity records: 21
Unique compounds: 21
Publication: ['CHEMBL3638553']
Source IDs: [37]
Source record count: 21

POTENTIAL DUPLICATE GROUP 2


,1712,5617
assay_chembl_id,CHEMBL3705522,CHEMBL5729936
assay_type,B,B
confidence_score,9,9
bao_label,single protein format,single protein format
description,In Vitro Kinase Assay: In vitro kinase analysis was performed by using HTScan EGF Receptor Kinas...,In Vitro Kinase Assay: In vitro kinase analysis was performed by using HTScan EGF Receptor Kinas...
document_chembl_id,CHEMBL3638441,CHEMBL3638441
src_id,37,37
src_assay_id,6599_1_pol_4952,6599_1_pol_520
target_chembl_id,CHEMBL203,CHEMBL203



Assay: CHEMBL3705522
Activity records: 6
Unique compounds: 6
Publication: ['CHEMBL3638441']
Source IDs: [37]
Source record count: 6

Assay: CHEMBL5729936
Activity records: 6
Unique compounds: 6
Publication: ['CHEMBL3638441']
Source IDs: [37]
Source record count: 6

POTENTIAL DUPLICATE GROUP 3


,1711,5616
assay_chembl_id,CHEMBL3705672,CHEMBL5729933
assay_type,B,B
confidence_score,9,9
bao_label,single protein format,single protein format
description,Time-Resolved Fluorescence Assay: The following assay may be used to determine the activity of t...,Time-Resolved Fluorescence Assay: The following assay may be used to determine the activity of t...
document_chembl_id,CHEMBL3639095,CHEMBL3639095
src_id,37,37
src_assay_id,6583_1_pol_4952,6583_1_pol_520
target_chembl_id,CHEMBL203,CHEMBL203



Assay: CHEMBL3705672
Activity records: 7
Unique compounds: 7
Publication: ['CHEMBL3639095']
Source IDs: [37]
Source record count: 7

Assay: CHEMBL5729933
Activity records: 7
Unique compounds: 7
Publication: ['CHEMBL3639095']
Source IDs: [37]
Source record count: 7

POTENTIAL DUPLICATE GROUP 4


,1708,5615
assay_chembl_id,CHEMBL3705968,CHEMBL5729875
assay_type,B,B
confidence_score,9,9
bao_label,single protein format,single protein format
description,ELISA-Based Kinase Assay: Inhibition of erbB tyrosine kinase activity was assessed using an ELIS...,ELISA-Based Kinase Assay: Inhibition of erbB tyrosine kinase activity was assessed using an ELIS...
document_chembl_id,CHEMBL3639119,CHEMBL3639119
src_id,37,37
src_assay_id,6153_1_pol_4952,6153_1_pol_520
target_chembl_id,CHEMBL203,CHEMBL203



Assay: CHEMBL3705968
Activity records: 7
Unique compounds: 7
Publication: ['CHEMBL3639119']
Source IDs: [37]
Source record count: 7

Assay: CHEMBL5729875
Activity records: 7
Unique compounds: 7
Publication: ['CHEMBL3639119']
Source IDs: [37]
Source record count: 7


In [ ]:
potential_pairs = [
    ("CHEMBL3705330", "CHEMBL5729840"),
    ("CHEMBL3705522", "CHEMBL5729936"),
    ("CHEMBL3705672", "CHEMBL5729933"),
    ("CHEMBL3705968", "CHEMBL5729875")
]

comparison_columns = [
    "molecule_chembl_id",
    "standard_value",
    "standard_units",
    "standard_relation",
    "record_id"
]

for assay_a, assay_b in potential_pairs:

    a = (
        assay_deduplicated[
            assay_deduplicated["assay_chembl_id"] == assay_a
        ][comparison_columns]
        .sort_values(comparison_columns)
        .reset_index(drop=True)
    )

    b = (
        assay_deduplicated[
            assay_deduplicated["assay_chembl_id"] == assay_b
        ][comparison_columns]
        .sort_values(comparison_columns)
        .reset_index(drop=True)
    )

    print("\n" + "=" * 65)
    print(f"{assay_a} vs {assay_b}")
    print("=" * 65)

    print("Records in A:", len(a))
    print("Records in B:", len(b))

    print(
        "Entire source measurement tables identical:",
        a.equals(b)
    )

    print(
        "Matching source record IDs:",
        len(set(a["record_id"]) & set(b["record_id"]))
    )

    print(
        "Matching compound IDs:",
        len(set(a["molecule_chembl_id"]) &
            set(b["molecule_chembl_id"]))
    )


CHEMBL3705330 vs CHEMBL5729840
Records in A: 21
Records in B: 21
Entire source measurement tables identical: True
Matching source record IDs: 21
Matching compound IDs: 21

CHEMBL3705522 vs CHEMBL5729936
Records in A: 6
Records in B: 6
Entire source measurement tables identical: True
Matching source record IDs: 6
Matching compound IDs: 6

CHEMBL3705672 vs CHEMBL5729933
Records in A: 7
Records in B: 7
Entire source measurement tables identical: True
Matching source record IDs: 7
Matching compound IDs: 7

CHEMBL3705968 vs CHEMBL5729875
Records in A: 7
Records in B: 7
Entire source measurement tables identical: True
Matching source record IDs: 7
Matching compound IDs: 7


In [ ]:
# Confirmed duplicate assay IDs to exclude

duplicate_assay_ids = [
    "CHEMBL5729840",
    "CHEMBL5729936",
    "CHEMBL5729933",
    "CHEMBL5729875"
]

before_count = len(assay_deduplicated)

assay_deduplicated = assay_deduplicated[
    ~assay_deduplicated["assay_chembl_id"].isin(
        duplicate_assay_ids
    )
].copy()

removed_count = before_count - len(assay_deduplicated)

print("CONFIRMED DUPLICATE ASSAY REMOVAL")
print("=" * 55)

print("Records before removal:", before_count)
print("Records removed:", removed_count)
print("Records remaining:", len(assay_deduplicated))
print("Unique assays remaining:",
      assay_deduplicated["assay_chembl_id"].nunique())

print("\nExcluded assay verification:")

for assay_id in duplicate_assay_ids:
    count = (
        assay_deduplicated["assay_chembl_id"] == assay_id
    ).sum()
    print(assay_id, ":", count)

CONFIRMED DUPLICATE ASSAY REMOVAL
Records before removal: 4523
Records removed: 41
Records remaining: 4482
Unique assays remaining: 533

Excluded assay verification:
CHEMBL5729840 : 0
CHEMBL5729936 : 0
CHEMBL5729933 : 0
CHEMBL5729875 : 0


In [ ]:
# Final integrity audit

final_df = assay_deduplicated.copy()

integrity_checks = {
    "Confidence score 9": final_df["confidence_score"].eq(9).all(),
    "Binding assay type": final_df["assay_type"].eq("B").all(),
    "Single protein format": final_df["bao_label"].eq(
        "single protein format"
    ).all(),
    "No explicit mutation annotations": (~final_df["mutation_flag"]).all(),
    "No ChEMBL validity warnings": final_df["data_validity_comment"].isna().all(),
    "No potential duplicate flags": final_df["potential_duplicate"].eq(0).all(),
    "Positive IC50 values": final_df["standard_value"].gt(0).all(),
    "Available canonical SMILES": final_df["canonical_smiles"].notna().all(),
    "Available pIC50": final_df["pIC50"].notna().all(),
    "IC50 measurement type": final_df["standard_type"].eq("IC50").all(),
    "Exact measurement relation": final_df["standard_relation"].eq("=").all(),
    "IC50 units in nM": final_df["standard_units"].eq("nM").all(),
    "Human target organism": final_df["target_organism"].eq("Homo sapiens").all(),
    "Unique activity IDs": final_df["activity_id"].is_unique
}

print("FINAL DATA INTEGRITY AUDIT")
print("=" * 55)

for check, result in integrity_checks.items():
    print(f"{check}: {'PASS' if result else 'FAIL'}")

print("\nOVERALL RESULT")
print("=" * 55)

if all(integrity_checks.values()):
    print("SUCCESS: All integrity checks passed.")
else:
    print("WARNING: Some checks failed. Investigate before proceeding.")

print("\nFINAL DATASET SUMMARY")
print("=" * 55)

print("Activity records:", len(final_df))
print("Unique compounds:", final_df["molecule_chembl_id"].nunique())
print("Unique assays:", final_df["assay_chembl_id"].nunique())
print("Unique publications:", final_df["document_chembl_id"].nunique())

print("\npIC50 distribution:")
display(final_df["pIC50"].describe())

FINAL DATA INTEGRITY AUDIT
Confidence score 9: PASS
Binding assay type: PASS
Single protein format: PASS
No explicit mutation annotations: PASS
No ChEMBL validity warnings: PASS
No potential duplicate flags: PASS
Positive IC50 values: PASS
Available canonical SMILES: PASS
Available pIC50: PASS
IC50 measurement type: PASS
Exact measurement relation: PASS
IC50 units in nM: PASS
Human target organism: PASS
Unique activity IDs: PASS

OVERALL RESULT
SUCCESS: All integrity checks passed.

FINAL DATASET SUMMARY
Activity records: 4482
Unique compounds: 4181
Unique assays: 533
Unique publications: 445

pIC50 distribution:


,pIC50
count,4482.000000
mean,7.242603
std,1.284400
min,4.000000
25%,6.259637
50%,7.365523
75%,8.301030
max,11.000000


### Additional Source-Level Duplicate Assay Removal

Four additional pairs of duplicate assay representations were identified through exact measurement-signature comparison.

The pairs were:

- CHEMBL3705330 / CHEMBL5729840
- CHEMBL3705522 / CHEMBL5729936
- CHEMBL3705672 / CHEMBL5729933
- CHEMBL3705968 / CHEMBL5729875

Each pair shared the same publication, source ID, compound identifiers, IC50 measurements, measurement units, relations, and source record IDs.

One representation from each pair was retained, while the duplicate representation was excluded to prevent repeated source measurements from receiving additional weight.

All original records were preserved in the unmodified source dataset.

In [68]:
final_df.drop(
    columns=["rdkit_mol", "smiles_valid"],
    errors="ignore"
).to_csv(
    "EGFR_final_curated_dataset.csv",
    index=False
)

In [69]:
from google.colab import files

files.download("EGFR_final_curated_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
# Check whether the curated dataset is available

if "final_df" in globals():
    print("Dataset already available!")
    print("Shape:", final_df.shape)

else:
    print("Dataset not found in runtime.")
    print("Please upload EGFR_final_curated_dataset.csv")

    from google.colab import files
    uploaded = files.upload()

    import pandas as pd

    final_df = pd.read_csv("EGFR_final_curated_dataset.csv")

    print("Dataset loaded successfully!")
    print("Shape:", final_df.shape)

Dataset not found in runtime.
Please upload EGFR_final_curated_dataset.csv


Saving EGFR_final_curated_dataset.csv to EGFR_final_curated_dataset.csv
Dataset loaded successfully!
Shape: (4482, 50)


In [2]:
!pip install -q rdkit

In [3]:
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors

print("RDKit version:", __import__("rdkit").__version__)
print("RDKit environment successfully initialized.")

RDKit version: 2026.09.2
RDKit environment successfully initialized.


In [4]:
# Create a separate structure-validation dataframe
structure_df = final_df.copy()

# Convert SMILES strings into RDKit molecular objects
structure_df["rdkit_mol"] = (
    structure_df["canonical_smiles"].apply(
        lambda smiles: Chem.MolFromSmiles(smiles)
    )
)

# Identify valid and invalid structures
structure_df["smiles_valid"] = (
    structure_df["rdkit_mol"].notna()
)

# Validation summary
print("MOLECULAR STRUCTURE VALIDATION")
print("=" * 50)

print("Total activity records:", len(structure_df))

print(
    "Unique compounds:",
    structure_df["molecule_chembl_id"].nunique()
)

print(
    "Valid SMILES records:",
    structure_df["smiles_valid"].sum()
)

print(
    "Invalid SMILES records:",
    (~structure_df["smiles_valid"]).sum()
)

print(
    "Unique compounds with valid SMILES:",
    structure_df.loc[
        structure_df["smiles_valid"],
        "molecule_chembl_id"
    ].nunique()
)

MOLECULAR STRUCTURE VALIDATION
Total activity records: 4482
Unique compounds: 4181
Valid SMILES records: 4482
Invalid SMILES records: 0
Unique compounds with valid SMILES: 4181


In [5]:
# Inspect five unique compounds

unique_structures = (
    structure_df[
        ["molecule_chembl_id", "canonical_smiles", "rdkit_mol"]
    ]
    .drop_duplicates(subset="molecule_chembl_id")
    .head(5)
    .copy()
)

for _, row in unique_structures.iterrows():

    mol = row["rdkit_mol"]

    print("\nCompound:", row["molecule_chembl_id"])
    print("SMILES:", row["canonical_smiles"])
    print("Number of atoms:", mol.GetNumAtoms())
    print("Number of bonds:", mol.GetNumBonds())
    print("Molecular formula:", rdMolDescriptors.CalcMolFormula(mol))


Compound: CHEMBL939
SMILES: COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1
Number of atoms: 31
Number of bonds: 34
Molecular formula: C22H24ClFN4O3

Compound: CHEMBL95320
SMILES: C#Cc1cccc(Nc2c(C#N)cnc3cc(OCCOC)c(OCCOC)cc23)c1
Number of atoms: 31
Number of bonds: 33
Molecular formula: C24H23N3O4

Compound: CHEMBL293261
SMILES: O=C(CCc1c(SSc2[nH]c3ccccc3c2CCC(=O)NCc2ccccc2)[nH]c2ccccc12)NCc1ccccc1
Number of atoms: 44
Number of bonds: 49
Molecular formula: C36H34N4O2S2

Compound: CHEMBL31965
SMILES: C=CC(=O)Nc1cc2c(Nc3ccc(F)c(Cl)c3)ncnc2cc1OCCCN1CCOCC1
Number of atoms: 34
Number of bonds: 37
Molecular formula: C24H25ClFN5O3

Compound: CHEMBL95774
SMILES: C=CC(=O)Nc1cc2c(Nc3ccc(F)c(Cl)c3)c(C#N)cnc2cc1OCCCN1CCOCC1
Number of atoms: 36
Number of bonds: 39
Molecular formula: C26H25ClFN5O3


In [6]:
# Analyse disconnected molecular fragments

unique_compounds = (
    structure_df[
        ["molecule_chembl_id", "canonical_smiles", "rdkit_mol"]
    ]
    .drop_duplicates(subset="molecule_chembl_id")
    .copy()
)

# Count fragments in each molecular structure
unique_compounds["fragment_count"] = (
    unique_compounds["rdkit_mol"].apply(
        lambda mol: len(Chem.GetMolFrags(mol))
    )
)

# Calculate the number of compounds with multiple fragments
single_fragment = (
    unique_compounds["fragment_count"] == 1
).sum()

multiple_fragments = (
    unique_compounds["fragment_count"] > 1
).sum()

print("DISCONNECTED FRAGMENT ANALYSIS")
print("=" * 50)

print("Total unique compounds:", len(unique_compounds))
print("Single-fragment compounds:", single_fragment)
print("Multiple-fragment compounds:", multiple_fragments)

print("\nFragment count distribution:")
print(
    unique_compounds["fragment_count"]
    .value_counts()
    .sort_index()
)

# Display examples of compounds with multiple fragments
multi_fragment_df = unique_compounds[
    unique_compounds["fragment_count"] > 1
]

print("\nMULTI-FRAGMENT COMPOUND EXAMPLES")
print("=" * 50)

for _, row in multi_fragment_df.head(10).iterrows():

    fragments = Chem.GetMolFrags(
        row["rdkit_mol"],
        asMols=True
    )

    print("\nCompound:", row["molecule_chembl_id"])
    print("Number of fragments:", len(fragments))

    for i, fragment in enumerate(fragments, start=1):
        print(
            f"Fragment {i}:",
            Chem.MolToSmiles(fragment)
        )

DISCONNECTED FRAGMENT ANALYSIS
Total unique compounds: 4181
Single-fragment compounds: 4125
Multiple-fragment compounds: 56

Fragment count distribution:
fragment_count
1    4125
2      52
3       3
4       1
Name: count, dtype: int64

MULTI-FRAGMENT COMPOUND EXAMPLES

Compound: CHEMBL555321
Number of fragments: 2
Fragment 1: Cl
Fragment 2: Clc1cccc(Nc2ncnc3ccccc23)c1

Compound: CHEMBL5416410
Number of fragments: 2
Fragment 1: Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(CCO)CC2)n1
Fragment 2: O

Compound: CHEMBL534948
Number of fragments: 2
Fragment 1: Cc1cccc(C)c1Nc1ncc(-c2ccc(OCC3CCCN(C)C3)cc2)n2cncc12
Fragment 2: Cl

Compound: CHEMBL2048911
Number of fragments: 2
Fragment 1: Cl
Fragment 2: O=C(NCCn1ccc2ncnc(Nc3ccc(Oc4cccc5sncc45)c(Cl)c3)c21)[C@@H]1CCCN1

Compound: CHEMBL2048910
Number of fragments: 2
Fragment 1: CC(C)(C)C(=O)NCCn1ccc2ncnc(Nc3ccc(Oc4cccc5sncc45)c(Cl)c3)c21
Fragment 2: CS(=O)(=O)O

Compound: CHEMBL2048907
Number of fragments: 2
Fragment 1: CC(C)(O)C(=O)NCCn1ccc2ncnc(

In [7]:
# Detailed audit of multi-fragment structures

fragment_audit = []

for _, row in multi_fragment_df.iterrows():

    fragments = Chem.GetMolFrags(
        row["rdkit_mol"],
        asMols=True
    )

    for i, fragment in enumerate(fragments, start=1):

        fragment_audit.append({
            "molecule_chembl_id": row["molecule_chembl_id"],
            "fragment_number": i,
            "fragment_smiles": Chem.MolToSmiles(fragment),
            "heavy_atoms": fragment.GetNumHeavyAtoms(),
            "molecular_formula": rdMolDescriptors.CalcMolFormula(fragment)
        })

fragment_audit_df = pd.DataFrame(fragment_audit)

print("FRAGMENT COMPOSITION SUMMARY")
print("=" * 50)

print("Total fragments:", len(fragment_audit_df))
print(
    "Unique compounds:",
    fragment_audit_df["molecule_chembl_id"].nunique()
)

print("\nFragment heavy-atom distribution:")
print(
    fragment_audit_df["heavy_atoms"]
    .value_counts()
    .sort_index()
)

print("\nDETAILED FRAGMENT AUDIT")
display(
    fragment_audit_df.sort_values(
        ["molecule_chembl_id", "heavy_atoms"],
        ascending=[True, False]
    ).head(25)
)

FRAGMENT COMPOSITION SUMMARY
Total fragments: 117
Unique compounds: 56

Fragment heavy-atom distribution:
heavy_atoms
1     38
5      2
7      5
8      2
11    14
18     1
22     1
23     1
24     1
25     1
26     2
27     3
28     6
29     3
30     4
33     2
34     2
35     2
36     2
37     8
38     1
39     3
40     3
41     2
42     3
43     2
44     2
48     1
Name: count, dtype: int64

DETAILED FRAGMENT AUDIT


,molecule_chembl_id,fragment_number,fragment_smiles,heavy_atoms,molecular_formula
20,CHEMBL1201179,1,CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5ccc...,40,C29H26ClFN4O4S
21,CHEMBL1201179,2,Cc1ccc(S(=O)(=O)O)cc1,11,C7H8O3S
22,CHEMBL1201179,3,Cc1ccc(S(=O)(=O)O)cc1,11,C7H8O3S
23,CHEMBL1201179,4,O,1,H2O
10,CHEMBL2048907,1,CC(C)(O)C(=O)NCCn1ccc2ncnc(Nc3ccc(Oc4cccc5sncc...,36,C25H23ClN6O3S
11,CHEMBL2048907,2,CS(=O)(=O)O,5,CH4O3S
8,CHEMBL2048910,1,CC(C)(C)C(=O)NCCn1ccc2ncnc(Nc3ccc(Oc4cccc5sncc...,36,C26H25ClN6O2S
9,CHEMBL2048910,2,CS(=O)(=O)O,5,CH4O3S
7,CHEMBL2048911,2,O=C(NCCn1ccc2ncnc(Nc3ccc(Oc4cccc5sncc45)c(Cl)c...,37,C26H24ClN7O2S
6,CHEMBL2048911,1,Cl,1,HCl


In [8]:
# Compare the two largest fragments per compound

fragment_comparison = []

for _, row in multi_fragment_df.iterrows():

    fragments = Chem.GetMolFrags(
        row["rdkit_mol"],
        asMols=True
    )

    fragment_sizes = sorted(
        [fragment.GetNumHeavyAtoms() for fragment in fragments],
        reverse=True
    )

    largest = fragment_sizes[0]
    second_largest = fragment_sizes[1]

    fragment_comparison.append({
        "molecule_chembl_id": row["molecule_chembl_id"],
        "fragment_count": len(fragments),
        "largest_fragment_atoms": largest,
        "second_largest_atoms": second_largest,
        "size_difference": largest - second_largest,
        "size_ratio": round(largest / second_largest, 2)
    })

fragment_comparison_df = pd.DataFrame(fragment_comparison)

print("PARENT STRUCTURE AMBIGUITY AUDIT")
print("=" * 55)

print(
    fragment_comparison_df[
        "size_ratio"
    ].describe()
)

print("\nCOMPOUNDS WITH CLOSELY SIZED FRAGMENTS")

display(
    fragment_comparison_df[
        fragment_comparison_df["size_ratio"] < 2
    ].sort_values("size_ratio")
)

print("\nALL MULTI-FRAGMENT COMPOUNDS")

display(
    fragment_comparison_df.sort_values(
        "size_ratio"
    )
)

PARENT STRUCTURE AMBIGUITY AUDIT
count    56.000000
mean     21.033214
std      13.019904
min       3.450000
25%       5.562500
50%      27.000000
75%      30.000000
max      37.000000
Name: size_ratio, dtype: float64

COMPOUNDS WITH CLOSELY SIZED FRAGMENTS


,molecule_chembl_id,fragment_count,largest_fragment_atoms,second_largest_atoms,size_difference,size_ratio



ALL MULTI-FRAGMENT COMPOUNDS


,molecule_chembl_id,fragment_count,largest_fragment_atoms,second_largest_atoms,size_difference,size_ratio
15,CHEMBL2437223,2,38,11,27,3.45
14,CHEMBL2437224,2,39,11,28,3.55
18,CHEMBL3436356,2,39,11,28,3.55
16,CHEMBL3436354,2,39,11,28,3.55
12,CHEMBL2437226,2,40,11,29,3.64
17,CHEMBL3436355,2,40,11,29,3.64
10,CHEMBL1201179,4,40,11,29,3.64
13,CHEMBL2437225,2,41,11,30,3.73
22,CHEMBL3436360,2,41,11,30,3.73
21,CHEMBL3436359,2,42,11,31,3.82


In [9]:
from rdkit.Chem.MolStandardize import rdMolStandardize

In [10]:
# Create a separate compound-level standardisation dataset

parent_df = unique_compounds.copy()

# Apply RDKit FragmentParent standardisation
parent_df["parent_mol"] = parent_df["rdkit_mol"].apply(
    lambda mol: rdMolStandardize.FragmentParent(mol)
)

# Generate standardised parent SMILES
parent_df["parent_smiles"] = parent_df["parent_mol"].apply(
    lambda mol: Chem.MolToSmiles(mol, canonical=True)
)

# Calculate parent molecular formula
parent_df["parent_formula"] = parent_df["parent_mol"].apply(
    rdMolDescriptors.CalcMolFormula
)

# Compare original and parent structures
parent_df["original_fragment_count"] = (
    parent_df["rdkit_mol"].apply(
        lambda mol: len(Chem.GetMolFrags(mol))
    )
)

parent_df["parent_fragment_count"] = (
    parent_df["parent_mol"].apply(
        lambda mol: len(Chem.GetMolFrags(mol))
    )
)

parent_df["structure_changed"] = (
    parent_df["canonical_smiles"] != parent_df["parent_smiles"]
)

print("PARENT STRUCTURE STANDARDISATION AUDIT")
print("=" * 55)

print("Total unique compounds:", len(parent_df))

print(
    "Compounds with changed SMILES:",
    parent_df["structure_changed"].sum()
)

print(
    "Compounds with unchanged SMILES:",
    (~parent_df["structure_changed"]).sum()
)

print(
    "Parents with multiple fragments:",
    (parent_df["parent_fragment_count"] > 1).sum()
)

print("\nSTANDARDISATION EXAMPLES")

display(
    parent_df.loc[
        parent_df["original_fragment_count"] > 1,
        [
            "molecule_chembl_id",
            "canonical_smiles",
            "parent_smiles",
            "original_fragment_count",
            "parent_fragment_count",
            "structure_changed"
        ]
    ].head(15)
)

PARENT STRUCTURE STANDARDISATION AUDIT
Total unique compounds: 4181
Compounds with changed SMILES: 57
Compounds with unchanged SMILES: 4124
Parents with multiple fragments: 0

STANDARDISATION EXAMPLES


,molecule_chembl_id,canonical_smiles,parent_smiles,original_fragment_count,parent_fragment_count,structure_changed
25,CHEMBL555321,Cl.Clc1cccc(Nc2ncnc3ccccc23)c1,Clc1cccc(Nc2ncnc3ccccc23)c1,2,1,True
327,CHEMBL5416410,Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C...,Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(C...,2,1,True
328,CHEMBL534948,Cc1cccc(C)c1Nc1ncc(-c2ccc(OCC3CCCN(C)C3)cc2)n2...,Cc1cccc(C)c1Nc1ncc(-c2ccc(OCC3CCCN(C)C3)cc2)n2...,2,1,True
575,CHEMBL2048911,Cl.O=C(NCCn1ccc2ncnc(Nc3ccc(Oc4cccc5sncc45)c(C...,O=C(NCCn1ccc2ncnc(Nc3ccc(Oc4cccc5sncc45)c(Cl)c...,2,1,True
576,CHEMBL2048910,CC(C)(C)C(=O)NCCn1ccc2ncnc(Nc3ccc(Oc4cccc5sncc...,CC(C)(C)C(=O)NCCn1ccc2ncnc(Nc3ccc(Oc4cccc5sncc...,2,1,True
579,CHEMBL2048907,CC(C)(O)C(=O)NCCn1ccc2ncnc(Nc3ccc(Oc4cccc5sncc...,CC(C)(O)C(=O)NCCn1ccc2ncnc(Nc3ccc(Oc4cccc5sncc...,2,1,True
621,CHEMBL2070045,COc1ccc2cc(-c3nn(C(C)C)c4ncnc(N)c34)ccc2c1.Cl,COc1ccc2cc(-c3nn(C(C)C)c4ncnc(N)c34)ccc2c1,2,1,True
622,CHEMBL2070048,CCOc1ccc2cc(-c3nn(C(C)C)c4ncnc(N)c34)ccc2c1.Cl,CCOc1ccc2cc(-c3nn(C(C)C)c4ncnc(N)c34)ccc2c1,2,1,True
730,CHEMBL2419762,Cl.c1ccc(-c2cccc(Nc3ncnc4cc5c(cc34)OCCCO5)c2)cc1,c1ccc(-c2cccc(Nc3ncnc4cc5c(cc34)OCCCO5)c2)cc1,2,1,True
731,CHEMBL2419760,Cl.c1ccc(-c2cccc(Nc3ncnc4cc5c(cc34)OCO5)c2)cc1,c1ccc(-c2cccc(Nc3ncnc4cc5c(cc34)OCO5)c2)cc1,2,1,True


In [11]:
# Find structures that changed despite having only one original fragment

single_fragment_changes = parent_df[
    (parent_df["original_fragment_count"] == 1)
    & (parent_df["structure_changed"])
].copy()

print("SINGLE-FRAGMENT STRUCTURES WITH CHANGES")
print("=" * 55)

print("Number of affected compounds:", len(single_fragment_changes))

display(
    single_fragment_changes[
        [
            "molecule_chembl_id",
            "canonical_smiles",
            "parent_smiles",
            "parent_formula"
        ]
    ]
)

SINGLE-FRAGMENT STRUCTURES WITH CHANGES
Number of affected compounds: 1


,molecule_chembl_id,canonical_smiles,parent_smiles,parent_formula
1479,CHEMBL3903713,C(=C/c1ccccc1)\C(=N\Nc1nncc2ccccc12)c1ccccc1,C(=C\c1ccccc1)/C(=N/Nc1nncc2ccccc12)c1ccccc1,C23H18N4


In [12]:
# Investigate stereochemical differences

compound_id = "CHEMBL3903713"

row = parent_df[
    parent_df["molecule_chembl_id"] == compound_id
].iloc[0]

original_mol = row["rdkit_mol"]
parent_mol = row["parent_mol"]

print("COMPOUND:", compound_id)
print("=" * 55)

print("Original SMILES:")
print(Chem.MolToSmiles(original_mol, canonical=True))

print("\nParent SMILES:")
print(Chem.MolToSmiles(parent_mol, canonical=True))

print("\nOriginal stereochemical assignments:")
print(Chem.FindMolChiralCenters(
    original_mol,
    includeUnassigned=True
))

print("\nParent stereochemical assignments:")
print(Chem.FindMolChiralCenters(
    parent_mol,
    includeUnassigned=True
))

print("\nOriginal double-bond stereochemistry:")

for bond in original_mol.GetBonds():
    if bond.GetBondType() == Chem.BondType.DOUBLE:
        if bond.GetStereo() != Chem.BondStereo.STEREONONE:
            print(
                bond.GetIdx(),
                bond.GetStereo()
            )

print("\nParent double-bond stereochemistry:")

for bond in parent_mol.GetBonds():
    if bond.GetBondType() == Chem.BondType.DOUBLE:
        if bond.GetStereo() != Chem.BondStereo.STEREONONE:
            print(
                bond.GetIdx(),
                bond.GetStereo()
            )

COMPOUND: CHEMBL3903713
Original SMILES:
C(=C\c1ccccc1)/C(=N/Nc1nncc2ccccc12)c1ccccc1

Parent SMILES:
C(=C\c1ccccc1)/C(=N/Nc1nncc2ccccc12)c1ccccc1

Original stereochemical assignments:
[]

Parent stereochemical assignments:
[]

Original double-bond stereochemistry:
0 STEREOE
8 STEREOZ

Parent double-bond stereochemistry:
0 STEREOE
8 STEREOZ


In [13]:
# Final parent-structure integrity checks

parent_df["parent_valid"] = parent_df["parent_smiles"].apply(
    lambda smiles: Chem.MolFromSmiles(smiles) is not None
)

parent_df["parent_heavy_atoms"] = parent_df["parent_mol"].apply(
    lambda mol: mol.GetNumHeavyAtoms()
)

integrity_summary = {
    "Total unique compounds": len(parent_df),
    "Valid parent SMILES": parent_df["parent_valid"].sum(),
    "Invalid parent SMILES": (~parent_df["parent_valid"]).sum(),
    "Single-fragment parents": (
        parent_df["parent_fragment_count"] == 1
    ).sum(),
    "Multi-fragment parents": (
        parent_df["parent_fragment_count"] > 1
    ).sum(),
    "Unique parent structures": parent_df["parent_smiles"].nunique()
}

print("FINAL PARENT STRUCTURE INTEGRITY AUDIT")
print("=" * 55)

for metric, value in integrity_summary.items():
    print(f"{metric}: {value}")

# Identify parent structures shared by multiple ChEMBL IDs
parent_duplicates = (
    parent_df.groupby("parent_smiles")
    .agg(
        compound_count=("molecule_chembl_id", "nunique"),
        compound_ids=("molecule_chembl_id", list)
    )
    .query("compound_count > 1")
    .sort_values("compound_count", ascending=False)
)

print("\nDUPLICATE PARENT STRUCTURE AUDIT")
print("=" * 55)

print("Groups with shared parent structures:", len(parent_duplicates))

display(parent_duplicates.head(20))

FINAL PARENT STRUCTURE INTEGRITY AUDIT
Total unique compounds: 4181
Valid parent SMILES: 4181
Invalid parent SMILES: 0
Single-fragment parents: 4181
Multi-fragment parents: 0
Unique parent structures: 4171

DUPLICATE PARENT STRUCTURE AUDIT
Groups with shared parent structures: 10


,compound_count,compound_ids
parent_smiles,,
C=CC(=O)Nc1cc2c(Nc3ccc(F)c(Cl)c3)ncnc2cc1OCCCN1CCOCC1,2,"[CHEMBL31965, CHEMBL545315]"
CN(C)C/C=C/C(=O)Nc1cc2c(Nc3ccc(F)c(Cl)c3)ncnc2cc1O[C@H]1CCOC1,2,"[CHEMBL1173655, CHEMBL2105712]"
CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5cccc(F)c5)c(Cl)c4)c3c2)o1,2,"[CHEMBL554, CHEMBL1201179]"
CS(=O)(=O)NCCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5cccc(F)c5)c(Cl)c4)c3c2)o1,2,"[CHEMBL2437225, CHEMBL3040597]"
CSCCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5cccc(F)c5)c(Cl)c4)c3c2)o1,2,"[CHEMBL2437223, CHEMBL3040455]"
C[S+]([O-])CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5cccc(F)c5)c(Cl)c4)c3c2)o1,2,"[CHEMBL2437224, CHEMBL3040548]"
NS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5cccc(F)c5)c(Cl)c4)c3c2)o1,2,"[CHEMBL2437226, CHEMBL3040566]"
O=S(=O)(CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5cccc(F)c5)c(Cl)c4)c3c2)o1)CC(F)(F)F,2,"[CHEMBL2437227, CHEMBL3040543]"
c1ccc(-c2cccc(Nc3ncnc4cc5c(cc34)OCCCO5)c2)cc1,2,"[CHEMBL2419762, CHEMBL3040910]"


In [14]:
# Map parent structures to the curated activity dataset

activity_parent_df = final_df.merge(
    parent_df[
        ["molecule_chembl_id", "parent_smiles"]
    ],
    on="molecule_chembl_id",
    how="left",
    validate="many_to_one"
)

print("ACTIVITY-LEVEL PARENT MAPPING AUDIT")
print("=" * 55)

print("Total activity records:", len(activity_parent_df))

print(
    "Records with missing parent SMILES:",
    activity_parent_df["parent_smiles"].isna().sum()
)

print(
    "Unique parent structures in activity dataset:",
    activity_parent_df["parent_smiles"].nunique()
)

# Investigate shared parent structures

shared_parent_ids = (
    parent_duplicates
    .reset_index()
)

shared_parent_audit = []

for _, row in shared_parent_ids.iterrows():

    ids = row["compound_ids"]

    subset = activity_parent_df[
        activity_parent_df["molecule_chembl_id"].isin(ids)
    ]

    shared_parent_audit.append({
        "compound_ids": ids,
        "activity_records": len(subset),
        "unique_assays": subset["assay_chembl_id"].nunique(),
        "unique_publications": subset["document_chembl_id"].nunique(),
        "minimum_pIC50": subset["pIC50"].min(),
        "maximum_pIC50": subset["pIC50"].max(),
        "mean_pIC50": subset["pIC50"].mean()
    })

shared_parent_audit_df = pd.DataFrame(
    shared_parent_audit
)

display(shared_parent_audit_df)

ACTIVITY-LEVEL PARENT MAPPING AUDIT
Total activity records: 4482
Records with missing parent SMILES: 0
Unique parent structures in activity dataset: 4171


,compound_ids,activity_records,unique_assays,unique_publications,minimum_pIC50,maximum_pIC50,mean_pIC50
0,"[CHEMBL31965, CHEMBL545315]",7,7,7,7.130768,9.337242,8.564894
1,"[CHEMBL1173655, CHEMBL2105712]",8,8,8,7.522879,10.096910,8.857307
2,"[CHEMBL554, CHEMBL1201179]",11,11,11,6.552842,8.259637,7.704008
3,"[CHEMBL2437225, CHEMBL3040597]",2,2,2,7.728158,7.808549,7.768354
4,"[CHEMBL2437223, CHEMBL3040455]",2,2,2,7.249955,7.600326,7.425140
5,"[CHEMBL2437224, CHEMBL3040548]",2,2,2,7.278437,7.886057,7.582247
6,"[CHEMBL2437226, CHEMBL3040566]",2,2,2,7.966576,8.567031,8.266803
7,"[CHEMBL2437227, CHEMBL3040543]",2,2,2,7.675718,7.835350,7.755534
8,"[CHEMBL2419762, CHEMBL3040910]",2,2,2,7.568636,7.568636,7.568636
9,"[CHEMBL2419760, CHEMBL3040861]",2,2,2,8.698970,8.698970,8.698970


In [15]:
# Create a unique parent-level mapping table

parent_mapping_df = (
    activity_parent_df[
        [
            "molecule_chembl_id",
            "parent_smiles"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["parent_smiles", "molecule_chembl_id"]
    )
    .reset_index(drop=True)
)

# Count ChEMBL IDs associated with each parent

parent_mapping_summary = (
    parent_mapping_df
    .groupby("parent_smiles")
    .agg(
        number_of_chembl_ids=(
            "molecule_chembl_id", "nunique"
        ),
        chembl_ids=(
            "molecule_chembl_id", list
        )
    )
    .reset_index()
)

print("PARENT IDENTITY MAPPING SUMMARY")
print("=" * 55)

print(
    "Unique parent structures:",
    len(parent_mapping_summary)
)

print(
    "Parent structures with multiple ChEMBL IDs:",
    (
        parent_mapping_summary["number_of_chembl_ids"] > 1
    ).sum()
)

print(
    "Maximum ChEMBL IDs per parent:",
    parent_mapping_summary["number_of_chembl_ids"].max()
)

print("\nShared parent structures:")

display(
    parent_mapping_summary[
        parent_mapping_summary["number_of_chembl_ids"] > 1
    ]
)

PARENT IDENTITY MAPPING SUMMARY
Unique parent structures: 4171
Parent structures with multiple ChEMBL IDs: 10
Maximum ChEMBL IDs per parent: 2

Shared parent structures:


,parent_smiles,number_of_chembl_ids,chembl_ids
604,C=CC(=O)Nc1cc2c(Nc3ccc(F)c(Cl)c3)ncnc2cc1OCCCN...,2,"[CHEMBL31965, CHEMBL545315]"
1754,CN(C)C/C=C/C(=O)Nc1cc2c(Nc3ccc(F)c(Cl)c3)ncnc2...,2,"[CHEMBL1173655, CHEMBL2105712]"
2775,CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5ccc...,2,"[CHEMBL1201179, CHEMBL554]"
2798,CS(=O)(=O)NCCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5cc...,2,"[CHEMBL2437225, CHEMBL3040597]"
2808,CSCCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5cccc(F)c5)c...,2,"[CHEMBL2437223, CHEMBL3040455]"
2874,C[S+]([O-])CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5cc...,2,"[CHEMBL2437224, CHEMBL3040548]"
3752,NS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5ccc...,2,"[CHEMBL2437226, CHEMBL3040566]"
4066,O=S(=O)(CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5cccc(...,2,"[CHEMBL2437227, CHEMBL3040543]"
4164,c1ccc(-c2cccc(Nc3ncnc4cc5c(cc34)OCCCO5)c2)cc1,2,"[CHEMBL2419762, CHEMBL3040910]"
4166,c1ccc(-c2cccc(Nc3ncnc4cc5c(cc34)OCO5)c2)cc1,2,"[CHEMBL2419760, CHEMBL3040861]"


## Phase 2 — Molecular Structure Validation and Standardisation

### Objective
To validate the molecular structures associated with the curated EGFR bioactivity dataset, investigate disconnected molecular fragments, and generate parent structures for consistent molecular representation.

### 1. SMILES Validation
All 4,181 unique ChEMBL compounds were successfully parsed using RDKit.

- Valid structures: 4,181
- Invalid structures: 0

### 2. Fragment Composition Audit
Disconnected fragments were investigated before parent selection.

- Single-fragment compounds: 4,125
- Multi-fragment compounds: 56
- Maximum observed fragment count: 4

The additional fragments included counterions, solvent molecules and other components. They were not automatically assumed to be errors.

### 3. Parent Structure Generation
RDKit `FragmentParent` was applied to generate parent molecular structures.

- Compounds processed: 4,181
- Multi-fragment parent structures remaining: 0
- Parent structures with valid SMILES: 4,181

A total of 56 compounds underwent fragment removal. One additional SMILES difference was confirmed to be a representation-only canonicalisation difference, with no change in the inspected stereochemistry.

### 4. Parent Identity Audit
The final dataset contained:

- Unique ChEMBL compound IDs: 4,181
- Unique parent structures: 4,171
- Shared parent structure groups: 10
- Missing parent mappings: 0

The 10 shared parent groups each contained two ChEMBL IDs. These records were retained to preserve experimental observations.

### 5. Data Integrity and Modelling Considerations
The original ChEMBL canonical SMILES and experimental activity measurements were preserved. Parent SMILES were stored separately.

For subsequent machine learning:

- Parent SMILES will serve as the molecular identity key.
- Records sharing a parent structure must remain in the same data partition.
- Shared parent records will not be blindly deleted or averaged.
- Molecular descriptors and fingerprints will be calculated from the parent structures.

### 6. Methodological Limitations
Parent generation used RDKit's `FragmentParent` standardisation procedure (RDKit version 2026.09.2).

This procedure does not constitute complete chemical standardisation. No separate tautomer canonicalisation or pH-dependent protonation-state normalisation was performed. These limitations will be considered when interpreting molecular representations and model predictions.

### Phase 2 Conclusion
All 4,181 unique compounds passed molecular parsing and parent-structure integrity checks. The resulting 4,171 unique parent structures provide the molecular foundation for subsequent descriptor calculation and molecular fingerprint generation.